# Paper 2 — Automatic 120-Scenario LDA + FP-Growth Ontology Pipeline (Colab v2.3)

This notebook runs the complete 5×4×2×3 validation sensitivity grid with resumable per-scenario checkpoints. Test-set execution is protected from parameter tuning.


## 1. Google Colab environment checks

In [ ]:
#@title Google Colab setup — single-scenario / resumable
from pathlib import Path
import os
import sys
import gc
import json
import shutil
import warnings
import platform

# ------------------------------------------------------------
# Clean console
# ------------------------------------------------------------
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning"
warnings.simplefilter("ignore", DeprecationWarning)

# Avoid CPU over-subscription.
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "2")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# ------------------------------------------------------------
# Colab paths
# ------------------------------------------------------------
COLAB_ROOT = Path("/content")
OUT = COLAB_ROOT / "paper2_outputs"
OUT.mkdir(parents=True, exist_ok=True)

# Optional persistent Google Drive mirror.
USE_GOOGLE_DRIVE = True
DRIVE_OUT = None

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(
            "/content/drive",
            force_remount=False,
        )

        DRIVE_OUT = (
            Path("/content/drive/MyDrive")
            / "Paper2_Experiments"
            / "paper2_outputs"
        )
        DRIVE_OUT.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            "Google Drive output:",
            DRIVE_OUT,
        )

    except Exception as exc:
        DRIVE_OUT = None
        print(
            "Google Drive mount unavailable; "
            "continuing with /content only.",
            repr(exc),
        )

print("Python      :", sys.version)
print("Platform    :", platform.platform())
print("Output dir  :", OUT)

# ------------------------------------------------------------
# GPU detection
# ------------------------------------------------------------
try:
    import torch

    DEVICE = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("Device      :", DEVICE)

    if torch.cuda.is_available():
        print(
            "GPU         :",
            torch.cuda.get_device_name(0),
        )

        total_vram = (
            torch.cuda
            .get_device_properties(0)
            .total_memory
            / 1024**3
        )

        print(
            f"GPU VRAM    : {total_vram:.2f} GB"
        )

except Exception as exc:
    DEVICE = "cpu"
    print(
        "GPU check warning:",
        repr(exc),
    )


def print_ram(label="RAM"):
    try:
        import psutil

        process = psutil.Process(
            os.getpid()
        )

        vm = psutil.virtual_memory()

        print(
            f"[{label}] "
            f"process={process.memory_info().rss/1024**3:.2f} GB | "
            f"available={vm.available/1024**3:.2f}/"
            f"{vm.total/1024**3:.2f} GB"
        )

    except Exception:
        pass


def cleanup_memory():
    gc.collect()

    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception:
        pass


def mirror_file(path):
    """
    Mirror one completed output/checkpoint to Google Drive when available.
    """
    if DRIVE_OUT is None:
        return

    path = Path(path)

    if not path.exists():
        return

    try:
        relative = path.relative_to(
            OUT
        )
    except Exception:
        relative = Path(
            path.name
        )

    destination = (
        DRIVE_OUT
        / relative
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        path,
        destination,
    )


def save_csv(df, path, **kwargs):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df.to_csv(
        path,
        **kwargs,
    )

    mirror_file(
        path
    )


def save_json(obj, path, **kwargs):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            **kwargs,
        )

    mirror_file(
        path
    )


def save_text(text, path, **kwargs):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    encoding = kwargs.pop(
        "encoding",
        "utf-8",
    )

    with open(
        path,
        "w",
        encoding=encoding,
        **kwargs,
    ) as f:
        f.write(
            text
        )

    mirror_file(
        path
    )


# ------------------------------------------------------------
# Checkpoint helpers
# ------------------------------------------------------------
CHECKPOINT_DIR = (
    OUT
    / "checkpoints"
)
CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def checkpoint_path(name):
    return (
        CHECKPOINT_DIR
        / name
    )


def checkpoint_exists(name):
    local_path = checkpoint_path(
        name
    )

    if local_path.exists():
        return True

    # Restore from Drive automatically if needed.
    if DRIVE_OUT is not None:
        drive_path = (
            DRIVE_OUT
            / "checkpoints"
            / name
        )

        if drive_path.exists():
            local_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            shutil.copy2(
                drive_path,
                local_path,
            )

            print(
                "[restored from Drive]",
                drive_path,
            )

            return True

    return False


def save_checkpoint_csv(df, name):
    path = checkpoint_path(
        name
    )

    save_csv(
        df,
        path,
        index=False,
    )

    print(
        f"[checkpoint saved] {path}"
    )

    return path


def load_checkpoint_csv(name):
    if not checkpoint_exists(
        name
    ):
        return None

    import pandas as pd

    path = checkpoint_path(
        name
    )

    df = pd.read_csv(
        path
    )

    print(
        f"[checkpoint loaded] "
        f"{path} | rows={len(df)}"
    )

    return df


print(
    "Checkpoint dir:",
    CHECKPOINT_DIR,
)

print_ram(
    "startup"
)


In [ ]:
#@title Resume saved results after Colab runtime crash — SELF-CONTAINED
from pathlib import Path
import shutil
import pandas as pd

OUT = Path("/content/paper2_outputs")
OUT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT = Path("/content/drive/MyDrive/Paper2_Experiments/paper2_outputs")

# Mount Drive when the new runtime has not mounted it yet.
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Google Drive mount warning:", repr(exc))

print("=" * 78)
print("SAVED SINGLE-SCENARIO RESULTS")
print("=" * 78)

patterns = [
    "single_scenario_results.csv",
    "single_scenario_results_validation.csv",
    "final_pareto_configurations.csv",
    "final_pareto_configurations_validation.csv",
    "feature_profile_topics*.csv",
    "scenario_details_*.csv",
]

restored = 0
if DRIVE_OUT.exists():
    for pattern in patterns:
        for drive_path in DRIVE_OUT.glob(pattern):
            local_path = OUT / drive_path.name
            if not local_path.exists() or drive_path.stat().st_mtime > local_path.stat().st_mtime:
                shutil.copy2(drive_path, local_path)
                restored += 1
                print("RESTORED |", drive_path.name)
else:
    print("Drive output folder not found:", DRIVE_OUT)

found = 0
for pattern in patterns:
    for path in sorted(OUT.glob(pattern)):
        found += 1
        try:
            rows = len(pd.read_csv(path))
        except Exception:
            rows = "?"
        print(f"FOUND    | {path.name:<60} | rows={rows}")

if found == 0:
    print("No saved scenario files found.")
else:
    main_file = OUT / "single_scenario_results.csv"
    if main_file.exists():
        saved_results = pd.read_csv(main_file)
        print("\nCompleted unique scenarios:", saved_results["configuration_id"].nunique())
        display(saved_results.tail())

print("=" * 78)
print("Files restored this run:", restored)
print(
    "After a crash, rerun setup/configuration, dataset loader, utilities, ROUGE, "
    "and original triple cache before running the next scenario."
)


## 2. Konfigurasi eksperimen

In [ ]:
# ============================================================
# KAGGLE NOTEBOOK SETUP
# ============================================================
from pathlib import Path
import os
import sys
import gc
import json
import platform

# Keep CPU-heavy libraries from over-subscribing Colab CPUs.
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "2")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

COLAB_WORKING = Path("/content")
OUT = COLAB_WORKING / "paper2_outputs"
OUT.mkdir(parents=True, exist_ok=True)

print("Python      :", sys.version)
print("Platform    :", platform.platform())
print("Colab work :", COLAB_WORKING)
print("Output dir  :", OUT)

def print_ram(label="RAM"):
    try:
        import psutil
        p = psutil.Process(os.getpid())
        vm = psutil.virtual_memory()
        print(
            f"[{label}] process={p.memory_info().rss/1024**3:.2f} GB | "
            f"available={vm.available/1024**3:.2f}/{vm.total/1024**3:.2f} GB"
        )
    except Exception:
        pass

def cleanup_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

def save_csv(df, path, **kwargs):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, **kwargs)

def save_json(obj, path, **kwargs):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, **kwargs)

def save_text(txt, path, **kwargs):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    encoding = kwargs.pop("encoding", "utf-8")
    with open(path, "w", encoding=encoding, **kwargs) as f:
        f.write(txt)

# ------------------------------------------------------------
# GPU detection
# ------------------------------------------------------------
try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    print("Device      :", DEVICE)

    if torch.cuda.is_available():
        print("GPU         :", torch.cuda.get_device_name(0))
        total_vram = (
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3
        )
        print(f"GPU VRAM    : {total_vram:.2f} GB")

except Exception as exc:
    DEVICE = "cpu"
    print("GPU check failed:", repr(exc))

print_ram("startup")


#@title Main configuration — Paper 1 aligned
from dataclasses import dataclass, asdict
from pathlib import Path
import os, random, json, warnings
import numpy as np

@dataclass
class Config:
    seed: int = 42
    run_mode: str = "FULL"       # DEMO or FULL
    output_dir: str = "/content/paper2_outputs"

    # Datasets
    summarisation_dataset: str = "Awesome075/multi_news_parquet"
    summarisation_split: str = "validation"
    strict_real_dataset: bool = True
    webnlg_dataset: str = "web_nlg"
    squad_dataset: str = "rajpurkar/squad"

    # NOTE: max_clusters = number of Multi-News document clusters loaded,
    # NOT the number of LDA topics.
    max_clusters: int = 5622
    max_webnlg: int = 1779
    max_squad: int = 10570

    # ----------------------------------------------------------------
    # PAPER 1 SUMMARISATION PARAMETERS
    # ----------------------------------------------------------------
    # Paper 1 reports 20 topics for DUC 2006 and 25 for DUC 2007.
    # Paper 2 uses Multi-News, so 20 is retained as the reference default.
    # Change to 25 only when reproducing the DUC 2007 setting.
    n_topics: int = 20
    lda_max_iter: int = 10
    lda_n_jobs: int = 1

    # Default single-run summary length. The sensitivity experiment below
    # iterates over summary_length_grid automatically.
    summary_sentences: int = 15

    # Downstream QA compares Paper-1-supported 10- and 15-sentence settings.
    # If a SQuAD context has fewer sentences than the requested budget,
    # generate_summary naturally returns all available sentences.
    qa_summary_length_grid: tuple = (10, 15)

    # FP-growth
    min_pattern_support: float = 0.08
    # Prevent combinatorial itemset explosion; lexical patterns use up to trigrams.
    max_pattern_length: int = 3
    # A frequent pattern must occur in at least two sentence transactions.
    # This prevents one-off combinations from being labelled as patterns in
    # very short clusters while retaining the configured proportional support.
    min_pattern_occurrences: int = 2
    # Deterministic topic-salience pruning before FP-Growth. This bounds the
    # combinatorial search without silently changing min_pattern_support.
    max_transaction_items: int = 20
    max_frequent_itemsets: int = 500_000
    fpgrowth_timeout_seconds: int = 300

    # Paper 1 equation:
    # Score(s) = alpha*TS(s) + beta*PR(s) - delta*Red(s,S)
    # The manuscript explicitly reports delta=5 but does not report numeric
    # alpha/beta values. Neutral alpha=beta=1 is therefore used here and
    # kept configurable. Replace them if the original Paper 1 source code
    # contains different values.
    alpha_topic: float = 1.0
    beta_pattern: float = 1.0
    redundancy_delta: float = 5.0


    # ============================================================
    # ONE-RUN PARAMETER TUNING
    # ============================================================
    # Stage A: tune α, β, δ using a fixed 15-sentence budget and
    # the reference n_topics setting. Alpha and beta are constrained
    # to sum to 1 so their interpretation remains clear.
    alpha_beta_grid: tuple = (
        (0.3, 0.7),
        (0.4, 0.6),
        (0.5, 0.5),
        (0.6, 0.4),
        (0.7, 0.3),
    )

    delta_grid: tuple = (1.0, 3.0, 5.0, 7.0)

    # Stage B: after selecting the best α,β,δ combination,
    # evaluate the Paper-1 sentence budgets.
    summary_length_grid: tuple = (10, 15)

    # Stage C: topic sensitivity after the best weighting/length is known.
    # 20 and 25 reproduce the Paper-1 DUC reference settings; 10 provides
    # a lower-complexity comparison.
    n_topics_grid: tuple = (10, 20, 25)


    # ============================================================
    # ROUGE STORAGE AND PARETO-BASED MODEL SELECTION
    # ============================================================
    # All standard ROUGE precision/recall/F1 values are persisted. ROUGE-SU4
    # recall is retained for comparability with Paper 1.
    saved_metrics: tuple = (
        "rouge1_precision", "rouge1_recall", "rouge1_f1",
        "rouge2_precision", "rouge2_recall", "rouge2_f1",
        "rougeL_precision", "rougeL_recall", "rougeL_f1",
        "rougeLsum_precision", "rougeLsum_recall", "rougeLsum_f1",
        "rougeSU4_precision", "rougeSU4_recall", "rougeSU4_f1",
        "preservation_f1",
        "compression_gain",
    )

    # Pareto uses non-redundant headline metrics. Recall and precision remain
    # available in every saved output but are not duplicate Pareto objectives.
    pareto_metrics: tuple = (
        "rouge1_f1",
        "rouge2_f1",
        "rougeL_f1",
        "rougeLsum_f1",
        "rougeSU4_recall",
        "preservation_f1",
        "compression_gain",
    )

    # Fixed setting used only in Stage A α–β–δ sensitivity.
    tuning_sentence_budget: int = 15

    # Candidate pool reported by Paper 1: top 27–50 sentences per topic.
    # 50 is used as the upper-bound setting.
    candidate_pool_per_topic: int = 50

    # SentenceTransformer is retained for downstream semantic QA/baseline
    # experiments, but it is NOT used by the Paper-1-aligned summariser.
    semantic_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    use_semantic_reranking: bool = True

    use_rebel: bool = False
    rebel_model: str = "Babelscape/rebel-large"
    save_intermediate: bool = True
    candidate_similarity_cap: int = 250
    cleanup_every_n_clusters: int = 1
    colab_smoke_test: bool = False

    # ============================================================
    # COLAB FREE / LOW-MEMORY EXECUTION
    # ============================================================
    low_memory_mode: bool = True
    checkpoint_resume: bool = True
    qa_batch_size: int = 250
    webnlg_batch_size: int = 250
    save_per_question_qa: bool = True
    keep_sentence_detail: bool = False
    max_sentences_for_similarity: int = 220

    # Automatic 120-scenario runner
    grid_checkpoint_every: int = 250
    grid_save_summary_text: bool = False
    grid_triple_cache_limit: int = 5000

CFG = Config()

# max_clusters is intentionally NOT overwritten.
# Set it explicitly to 5, 10, 15, 20, or 25 for each run.
if CFG.run_mode.upper() == "FULL":
    CFG.max_webnlg = 1779
    CFG.max_squad = 10570


random.seed(CFG.seed)
np.random.seed(CFG.seed)
os.environ["PYTHONHASHSEED"] = str(CFG.seed)
warnings.filterwarnings("ignore")

save_json(asdict(CFG), OUT / "config.json", indent=2)

print(json.dumps(asdict(CFG), indent=2))
print(
    "\nPaper-1 summariser:",
    f"alpha={CFG.alpha_topic}, beta={CFG.beta_pattern}, "
    f"delta={CFG.redundancy_delta}, topics={CFG.n_topics}, "
    f"summary_sentences={CFG.summary_sentences}"
)

# Final Colab output-path safeguard
OUT = Path("/content/paper2_outputs")
OUT.mkdir(parents=True, exist_ok=True)


# ============================================================
# RESUMABLE CHECKPOINT HELPERS
# ============================================================
CHECKPOINT_DIR = OUT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_path(name):
    return CHECKPOINT_DIR / name

def checkpoint_exists(name):
    return checkpoint_path(name).exists()

def save_checkpoint_csv(df, name):
    path = checkpoint_path(name)
    df.to_csv(path, index=False)
    print(f"[checkpoint saved] {path}")
    return path

def load_checkpoint_csv(name):
    import pandas as pd
    path = checkpoint_path(name)
    if not path.exists():
        return None
    df = pd.read_csv(path)
    print(f"[checkpoint loaded] {path} | rows={len(df)}")
    return df

print("Checkpoint dir:", CHECKPOINT_DIR)


**Kaggle/Python 3.12 note.** `jupyter_client` may emit a legacy UTC deprecation message from the notebook infrastructure itself rather than from the Paper-2 code. Version v11 suppresses `DeprecationWarning` globally while leaving errors, exceptions, `RuntimeWarning`, and `UserWarning` visible.


In [ ]:
# Defensive suppression for warnings emitted while this cell is active.
import warnings
warnings.simplefilter("ignore", DeprecationWarning)

#@title FAST resume status after Colab restart — metadata only
from pathlib import Path
import pandas as pd

print("=" * 78)
print("CHECKPOINT STATUS — FAST METADATA CHECK")
print("=" * 78)

checkpoint_files = [
    "triples_full.csv",
    "final_pareto_configurations.csv",
    "final_pareto_configurations_validation.csv",
    "webnlg_eval.csv",
    "squad_qa_detail.csv",
    "squad_qa_aggregate.csv",
]

available_checkpoints = {}

for name in checkpoint_files:
    path = checkpoint_path(name)
    exists = path.exists()

    if exists:
        size_mb = path.stat().st_size / (1024 ** 2)
        available_checkpoints[name] = path

        print(
            f"FOUND   | {name:<36} | "
            f"{size_mb:9.2f} MB"
        )
    else:
        print(
            f"missing | {name:<36}"
        )

print("=" * 78)

print(
    "\nFAST START MODE:"
    "\n- This cell DOES NOT load large CSV checkpoints into RAM."
    "\n- Checkpoints are loaded lazily only when the corresponding stage needs them."
    "\n- This prevents SQuAD/WebNLG checkpoint loading from delaying notebook startup."
)


# ============================================================
# Small checkpoint only: safe to restore immediately
# ============================================================
_pareto_name = "final_pareto_configurations.csv"

if checkpoint_exists(_pareto_name):
    _pareto_path = checkpoint_path(_pareto_name)

    # Final Pareto table is normally tiny, so restoring it here is cheap.
    FINAL_PARETO_CONFIGS = pd.read_csv(
        _pareto_path
    )

    print(
        f"\nRestored small Pareto checkpoint: "
        f"{len(FINAL_PARETO_CONFIGS)} rows"
    )


# ============================================================
# LAZY LOAD HELPERS
# ============================================================
def restore_triples_if_needed():
    global triples_df

    if (
        "triples_df" in globals()
        and isinstance(triples_df, pd.DataFrame)
        and not triples_df.empty
    ):
        return triples_df

    path = checkpoint_path(
        "triples_full.csv"
    )

    if not path.exists():
        return None

    print(
        "Loading triples checkpoint only when needed..."
    )

    triples_df = pd.read_csv(
        path
    )

    print(
        f"Loaded triples: {len(triples_df)} rows"
    )

    return triples_df


def restore_webnlg_if_needed():
    global web_eval

    if (
        "web_eval" in globals()
        and isinstance(web_eval, pd.DataFrame)
        and not web_eval.empty
    ):
        return web_eval

    path = checkpoint_path(
        "webnlg_eval.csv"
    )

    if not path.exists():
        return None

    print(
        "Loading WebNLG checkpoint only when needed..."
    )

    web_eval = pd.read_csv(
        path
    )

    print(
        f"Loaded WebNLG evaluation: {len(web_eval)} rows"
    )

    return web_eval


def restore_squad_if_needed():
    global qa_comparison_df
    global qa_aggregate

    detail_path = checkpoint_path(
        "squad_qa_detail.csv"
    )

    aggregate_path = checkpoint_path(
        "squad_qa_aggregate.csv"
    )

    if (
        (
            "qa_comparison_df" not in globals()
            or not isinstance(
                qa_comparison_df,
                pd.DataFrame,
            )
        )
        and detail_path.exists()
    ):
        print(
            "Loading detailed SQuAD checkpoint only when needed..."
        )

        qa_comparison_df = pd.read_csv(
            detail_path
        )

        print(
            f"Loaded SQuAD detail: "
            f"{len(qa_comparison_df)} rows"
        )

    if (
        (
            "qa_aggregate" not in globals()
            or not isinstance(
                qa_aggregate,
                pd.DataFrame,
            )
        )
        and aggregate_path.exists()
    ):
        qa_aggregate = pd.read_csv(
            aggregate_path
        )

        print(
            f"Loaded SQuAD aggregate: "
            f"{len(qa_aggregate)} rows"
        )

    return (
        globals().get(
            "qa_comparison_df"
        ),
        globals().get(
            "qa_aggregate"
        ),
    )


print(
    "\nResume metadata check complete."
)


In [ ]:
#@title Required package setup — installs only missing packages
import importlib
import importlib.util
import subprocess
import sys

# sentence-transformers is intentionally excluded because it is optional and
# the Paper-1-aligned LDA + FP-Growth summariser does not require it.
REQUIRED_PACKAGES = {
    "datasets": "datasets",
    "rouge_score": "rouge-score",
    "rdflib": "rdflib",
    "psutil": "psutil",
    "networkx": "networkx",
}

missing = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--disable-pip-version-check", "--no-input", *missing,
        ])
        importlib.invalidate_caches()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Package installation failed. Ensure Internet is enabled for this "
            "Colab/Kaggle session, rerun this setup cell, and only then continue."
        ) from exc

# Pin mlxtend and install it without dependency resolution. Colab already
# provides compatible NumPy, pandas, SciPy, scikit-learn, matplotlib and joblib.
# This prevents pip from replacing Colab's pinned pandas version.
if importlib.util.find_spec("mlxtend") is None:
    print("Installing mlxtend 0.23.4 without changing Colab/Kaggle dependencies...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--disable-pip-version-check", "--no-input", "--no-deps",
            "mlxtend==0.23.4",
        ])
        importlib.invalidate_caches()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "mlxtend installation failed. Enable Internet, then rerun this setup cell."
        ) from exc

still_missing = [
    module_name
    for module_name in [*REQUIRED_PACKAGES, "mlxtend"]
    if importlib.util.find_spec(module_name) is None
]
if still_missing:
    raise RuntimeError(
        "Packages remain unavailable after installation: "
        + ", ".join(still_missing)
        + ". Restart the runtime once, then rerun from the first cell."
    )

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
import mlxtend

print("All required packages are ready.")
print("mlxtend version:", mlxtend.__version__)


In [ ]:
#@title spaCy model check — CPU-SAFE (PyTorch optional)
import importlib
import sys

# spaCy only needs its CPU backend here. If the optional torch installation is
# broken, hide it while importing spaCy so Thinc does not initialize torch.
TORCH_AVAILABLE = False
try:
    import torch
    _ = torch.__version__
    TORCH_AVAILABLE = True
except Exception as exc:
    print("Optional PyTorch unavailable; continuing with CPU-only spaCy:", repr(exc))
    for module_name in list(sys.modules):
        if module_name == "torch" or module_name.startswith("torch."):
            sys.modules.pop(module_name, None)
    sys.modules["torch"] = None

try:
    import spacy
finally:
    # Remove the temporary sentinel after spaCy/Thinc has selected CPU mode.
    if sys.modules.get("torch") is None:
        sys.modules.pop("torch", None)

try:
    nlp_test = spacy.load("en_core_web_sm", disable=["ner"])
    print("spaCy model en_core_web_sm is available (CPU mode).")
    del nlp_test
except OSError as exc:
    raise RuntimeError(
        "Missing spaCy model en_core_web_sm. Run once: "
        "%pip install -q --no-deps "
        "https://github.com/explosion/spacy-models/releases/download/"
        "en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
    ) from exc


## Colab stability diagnostics

Candidate-only similarity, lazy semantic model loading, deferred SQuAD loading, RAM monitoring, cleanup between clusters/stages, and efficient immediate Drive mirroring are enabled.


In [ ]:
#@title Preflight resource check
print_ram('preflight')
try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('GPU check:', repr(exc))


## 3. Import, perangkat, dan utilitas reproduksibilitas

In [ ]:
#@title Runtime binary compatibility check
import subprocess
import sys

# Test real imports, not only package metadata.
check = """
import numpy, scipy, sklearn, pandas, pyarrow
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse
x = TfidfVectorizer().fit_transform(['alpha beta', 'beta delta'])
assert sparse.issparse(x)
assert cosine_similarity(x).shape == (2, 2)
print({
    'numpy': numpy.__version__,
    'scipy': scipy.__version__,
    'scikit-learn': sklearn.__version__,
    'pandas': pandas.__version__,
    'pyarrow': pyarrow.__version__,
})
"""
result = subprocess.run(
    [sys.executable, "-c", check],
    text=True, capture_output=True,
)
if result.returncode != 0:
    raise RuntimeError(
        "Scientific Python stack tidak konsisten. "
        "Gunakan Runtime > Disconnect and delete runtime, lalu buka "
        "notebook terbaru dan pilih Runtime > Run all.\n\n"
        + result.stderr
    )
print("Core binary compatibility: OK")
print(result.stdout.strip())


In [ ]:
import re
import random
import warnings
import numpy as np
import pandas as pd
import sys
try:
    import torch
    _ = torch.__version__
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    torch = None
    for module_name in list(sys.modules):
        if module_name == "torch" or module_name.startswith("torch."):
            sys.modules.pop(module_name, None)
    print("PyTorch optional import skipped; CPU pipeline remains available:", repr(exc))

from pathlib import Path
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

# ============================================================
# 1. PENGATURAN PERANGKAT DAN MODEL
# ============================================================

DEVICE = "cuda" if TORCH_AVAILABLE and torch.cuda.is_available() else "cpu"
SEED = getattr(CFG, "seed", 42)
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Device: {DEVICE} | Seed: {SEED}")

# Nilai default apabila atribut belum tersedia di CFG
USE_SEMANTIC_RERANKING = getattr(
    CFG,
    "use_semantic_reranking",
    True
)

SEMANTIC_MODEL_NAME = getattr(
    CFG,
    "semantic_model",
    "sentence-transformers/all-MiniLM-L6-v2"
)

SEMANTIC_THRESHOLD = getattr(
    CFG,
    "semantic_threshold",
    0.78
)

SAVE_INTERMEDIATE = getattr(
    CFG,
    "save_intermediate",
    True
)

SUMMARY_SENTENCES = getattr(
    CFG,
    "summary_sentences",
    5
)

N_TOPICS = getattr(
    CFG,
    "n_topics",
    5
)

SEED = getattr(
    CFG,
    "seed",
    42
)

MIN_PATTERN_SUPPORT = getattr(
    CFG,
    "min_pattern_support",
    2
)

# Pastikan output directory tersedia
OUT = Path("/content/paper2_outputs")
OUT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. SEMANTIC MODEL — LAZY/DISABLED FOR COLAB STABILITY
# ============================================================
# Paper-1 summarisation does not require SentenceTransformer.
# QA automatically falls back to TF-IDF when SEM_MODEL is None.
SEM_MODEL = None
print("SentenceTransformer eager loading skipped for Colab stability.")

# ============================================================
# 3. FUNGSI UTILITAS
# ============================================================

def minmax(x):
    """
    Normalisasi min-max yang aman terhadap:
    - array kosong;
    - NaN;
    - nilai konstan.
    """
    x = np.asarray(x, dtype=float)

    if x.size == 0:
        return x

    x = np.nan_to_num(
        x,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    x_min = np.min(x)
    x_max = np.max(x)

    if np.isclose(x_max, x_min):
        return np.zeros_like(x, dtype=float)

    return (x - x_min) / (x_max - x_min)


def safe_filename(value):
    """
    Membersihkan cluster_id agar aman dijadikan nama file.
    """
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    return value[:150]


def ensure_document_list(documents):
    """
    Memastikan input documents selalu berbentuk list string.
    """
    if documents is None:
        return []

    if isinstance(documents, str):
        return [documents]

    if isinstance(documents, (list, tuple)):
        return [
            str(doc)
            for doc in documents
            if doc is not None and str(doc).strip()
        ]

    try:
        return [
            str(doc)
            for doc in list(documents)
            if doc is not None and str(doc).strip()
        ]
    except Exception:
        return [str(documents)]


def safe_sentencize(documents):
    """
    Memecah seluruh dokumen menjadi kalimat,
    sambil membuang kalimat kosong dan sangat pendek.
    """
    documents = ensure_document_list(documents)

    sentences = []

    for document in documents:
        if not document or not document.strip():
            continue

        try:
            extracted = sentencize(document)
        except Exception:
            # Fallback sederhana jika fungsi sentencize bermasalah
            extracted = re.split(
                r"(?<=[.!?])\s+",
                document.strip()
            )

        for sentence in extracted:
            sentence = str(sentence).strip()

            # Hindari kalimat kosong atau terlalu pendek
            if len(sentence.split()) >= 3:
                sentences.append(sentence)

    return sentences


def calculate_centrality(sentences):
    """
    Menghitung centrality TF-IDF dengan fallback.
    """
    if not sentences:
        return np.array([], dtype=float)

    if len(sentences) == 1:
        return np.array([1.0], dtype=float)

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            lowercase=True,
            min_df=1,
            token_pattern=r"(?u)\b\w\w+\b"
        )

        tfidf = vectorizer.fit_transform(sentences)

        # sparse centroid tetap kompatibel dengan cosine_similarity
        centroid = np.asarray(tfidf.mean(axis=0))

        centrality = cosine_similarity(
            tfidf,
            centroid
        ).ravel()

        return np.nan_to_num(
            centrality,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

    except ValueError as e:
        # Contoh: empty vocabulary
        print("TF-IDF warning:", str(e))

        # Fallback berdasarkan panjang kalimat
        lengths = np.array(
            [len(sentence.split()) for sentence in sentences],
            dtype=float
        )

        return minmax(lengths)

    except Exception as e:
        print("Centrality warning:", repr(e))
        return np.ones(len(sentences), dtype=float)


def safe_topic_scores(sentences, ablation):
    """
    Topic score dengan fallback apabila LDA/topic modelling gagal.
    """
    if ablation == "no_topic":
        return np.zeros(len(sentences), dtype=float)

    try:
        result = topic_scores(
            sentences,
            N_TOPICS,
            SEED
        )

        result = np.asarray(result, dtype=float)

        if len(result) != len(sentences):
            raise ValueError(
                "Jumlah topic score tidak sama dengan jumlah kalimat."
            )

        return np.nan_to_num(result)

    except Exception as e:
        print("Topic scoring warning:", repr(e))
        return np.zeros(len(sentences), dtype=float)


def safe_pattern_scores(sentences, ablation):
    """
    Pattern score dengan fallback.
    """
    if ablation == "no_pattern":
        return (
            np.zeros(len(sentences), dtype=float),
            set()
        )

    try:
        scores, patterns = pattern_scores(
            sentences,
            MIN_PATTERN_SUPPORT
        )

        scores = np.asarray(scores, dtype=float)

        if len(scores) != len(sentences):
            raise ValueError(
                "Jumlah pattern score tidak sama dengan jumlah kalimat."
            )

        return np.nan_to_num(scores), patterns

    except Exception as e:
        print("Pattern scoring warning:", repr(e))

        return (
            np.zeros(len(sentences), dtype=float),
            set()
        )


def encode_sentences(sentences):
    """
    Menghasilkan embedding kalimat dengan batching dan fallback.
    """
    if SEM_MODEL is None:
        return None

    if not sentences:
        return None

    try:
        embeddings = SEM_MODEL.encode(
            sentences,
            batch_size=32,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        return np.asarray(
            embeddings,
            dtype=np.float32
        )

    except RuntimeError as e:
        # Umumnya CUDA out of memory
        print("Embedding runtime warning:", repr(e))

        if TORCH_AVAILABLE and torch.cuda.is_available():
            torch.cuda.empty_cache()

        try:
            # Coba ulang dengan batch kecil
            embeddings = SEM_MODEL.encode(
                sentences,
                batch_size=8,
                show_progress_bar=False,
                normalize_embeddings=True,
                convert_to_numpy=True
            )

            return np.asarray(
                embeddings,
                dtype=np.float32
            )

        except Exception as retry_error:
            print(
                "Embedding retry failed:",
                repr(retry_error)
            )
            return None

    except Exception as e:
        print("Embedding warning:", repr(e))
        return None


# ============================================================
# 4. FUNGSI GENERATE SUMMARY
# ============================================================

def generate_summary(
    documents,
    k=5,
    ablation="full",
    semantic_threshold=SEMANTIC_THRESHOLD
):
    """
    Membuat extractive summary berbasis:
    - topic score;
    - pattern score;
    - TF-IDF centrality;
    - semantic redundancy filtering.
    """

    sentences = safe_sentencize(documents)

    if not sentences:
        empty_detail = pd.DataFrame(
            columns=[
                "sentence",
                "topic_score",
                "pattern_score",
                "centrality",
                "final_score",
                "selected"
            ]
        )

        return "", empty_detail

    # Jangan memilih lebih banyak dari jumlah kalimat
    k = max(1, min(int(k), len(sentences)))

    topic = safe_topic_scores(
        sentences,
        ablation
    )

    pattern, patterns = safe_pattern_scores(
        sentences,
        ablation
    )

    centrality = calculate_centrality(sentences)

    # Pastikan semuanya panjangnya sama
    n_sentences = len(sentences)

    topic = np.resize(topic, n_sentences)
    pattern = np.resize(pattern, n_sentences)
    centrality = np.resize(centrality, n_sentences)

    final_score = (
        0.40 * minmax(topic)
        + 0.30 * minmax(pattern)
        + 0.30 * minmax(centrality)
    )

    # Stable sort agar hasil reproducible saat score sama
    ranked_indices = np.argsort(
        -final_score,
        kind="stable"
    )

    embeddings = None

    if (
        ablation != "no_semantic"
        and USE_SEMANTIC_RERANKING
    ):
        embeddings = encode_sentences(sentences)

    chosen = []

    for idx in ranked_indices:
        idx = int(idx)

        if len(chosen) >= k:
            break

        # Filter redundansi semantik
        if (
            embeddings is not None
            and chosen
            and ablation != "no_semantic"
        ):
            similarities = [
                float(
                    np.dot(
                        embeddings[idx],
                        embeddings[selected_idx]
                    )
                )
                for selected_idx in chosen
            ]

            redundancy = max(similarities)

            if redundancy > semantic_threshold:
                continue

        chosen.append(idx)

    # Jika filter terlalu ketat dan jumlah kalimat kurang dari k,
    # isi dengan kandidat berikutnya
    if len(chosen) < k:
        for idx in ranked_indices:
            idx = int(idx)

            if idx not in chosen:
                chosen.append(idx)

            if len(chosen) >= k:
                break

    # Urutkan kembali sesuai posisi asli dokumen
    chosen = sorted(chosen)

    summary = " ".join(
        sentences[idx]
        for idx in chosen
    )

    selected_set = set(chosen)

    detail = pd.DataFrame({
        "sentence_id": list(range(n_sentences)),
        "sentence": sentences,
        "topic_score": topic,
        "pattern_score": pattern,
        "centrality": centrality,
        "final_score": final_score,
        "selected": [
            idx in selected_set
            for idx in range(n_sentences)
        ]
    })

    detail["ablation"] = ablation
    detail["semantic_threshold"] = semantic_threshold
    detail["patterns_found"] = len(patterns)

    return summary, detail


# Eksekusi dataset dilakukan pada Bagian 6 setelah summary_data dimuat.
print("Import, device setup, and summarisation utilities are ready.")


In [ ]:
from datasets import load_dataset

print("load_dataset successfully imported; no additional installation was performed.")


In [ ]:
from pathlib import Path
import shutil

cache_paths = [
    Path("/content/paper2_v3_cache"),
    Path("/content/drive/MyDrive/paper2_colab_v3/cache")
]

for path in cache_paths:
    if path.exists():
        shutil.rmtree(path)
        print("Deleted:", path)


In [ ]:
#@title Deferred SQuAD loading
# SQuAD is loaded only in the downstream QA section to reduce startup memory/network pressure.
test_squad = None
print("SQuAD loading deferred until QA evaluation.")


## 4. Real Multi-News loader

In [ ]:
#@title Multi-News loader utility — DEFINE ONLY (empty-record safe)
from datasets import load_dataset
import pandas as pd


MULTINEWS_SOURCE_INFO = {
    "dataset_name": "Multi-News",
    "dataset_source": "Awesome075/multi_news_parquet",
    "split": None,
    "requested_clusters": None,
    "raw_split_records": None,
    "raw_records_scanned": None,
    "loaded_clusters": None,
    "skipped_records": None,
    "is_real_multinews": False,
    "fallback_demo_used": False,
}
MULTINEWS_SKIPPED_RECORDS = []


def _clean_multinews_text(value):
    """Normalize whitespace without changing dataset content semantics."""
    if value is None:
        return ""
    text = str(value).replace("\u00a0", " ")
    return " ".join(text.split())


def load_summarisation_data(limit: int):
    """Load real Multi-News and deterministically exclude unusable records.

    `limit` denotes raw records selected from the beginning of the official test
    split. Empty source/reference records are never fabricated: they are logged
    and excluded from evaluation, preserving an auditable denominator.
    """
    global MULTINEWS_SKIPPED_RECORDS

    limit = int(limit)
    if limit < 1:
        raise ValueError("limit must be at least 1")

    dataset_name = "Awesome075/multi_news_parquet"
    split_name = str(getattr(CFG, "summarisation_split", "validation")).lower().strip()
    if split_name not in {"train", "validation", "test"}:
        raise ValueError(f"Unsupported Multi-News split: {split_name}")
    MULTINEWS_SKIPPED_RECORDS = []

    MULTINEWS_SOURCE_INFO.update({
        "split": split_name,
        "requested_clusters": limit,
        "raw_split_records": None,
        "raw_records_scanned": None,
        "loaded_clusters": None,
        "skipped_records": None,
        "is_real_multinews": False,
        "fallback_demo_used": False,
    })

    print(f"Loading REAL Multi-News from {dataset_name} | split={split_name}")
    try:
        ds = load_dataset(dataset_name, split=split_name)
    except Exception as exc:
        raise RuntimeError(
            "REAL Multi-News Parquet could not be loaded.\n"
            f"Dataset: {dataset_name}\nSplit: {split_name}\nError: {repr(exc)}"
        ) from exc

    raw_split_records = len(ds)
    raw_records_scanned = min(limit, raw_split_records)
    if limit > raw_split_records:
        print(
            f"WARNING: requested {limit} raw records, but the split contains "
            f"only {raw_split_records}; scanning the complete split."
        )

    rows = []
    for i in range(raw_records_scanned):
        record = ds[i]
        if "document" not in record or "summary" not in record:
            raise RuntimeError(
                "Unexpected Multi-News schema. Expected fields 'document' and "
                f"'summary'. Available fields: {list(record.keys())}"
            )

        raw_document = _clean_multinews_text(record.get("document", ""))
        reference_summary = _clean_multinews_text(record.get("summary", ""))
        documents = [
            cleaned
            for part in raw_document.split("|||||")
            if (cleaned := _clean_multinews_text(part))
        ]

        reasons = []
        if not documents:
            reasons.append("empty_source_documents")
        if not reference_summary:
            reasons.append("empty_reference_summary")

        if reasons:
            MULTINEWS_SKIPPED_RECORDS.append({
                "raw_record_index": i,
                "cluster_id": f"mn_{i:04d}",
                "reason": ";".join(reasons),
                "dataset_source": dataset_name,
                "dataset_split": split_name,
            })
            continue

        rows.append({
            "cluster_id": f"mn_{i:04d}",
            "raw_record_index": i,
            "documents": documents,
            "reference_summary": reference_summary,
            "dataset_name": "Multi-News",
            "dataset_source": dataset_name,
            "dataset_split": split_name,
            "is_real_multinews": True,
            "source_document_count": len(documents),
        })

    if not rows:
        raise RuntimeError("Multi-News loader returned zero usable records.")

    MULTINEWS_SOURCE_INFO.update({
        "raw_split_records": raw_split_records,
        "raw_records_scanned": raw_records_scanned,
        "loaded_clusters": len(rows),
        "skipped_records": len(MULTINEWS_SKIPPED_RECORDS),
        "is_real_multinews": True,
    })

    if MULTINEWS_SKIPPED_RECORDS:
        print(
            f"WARNING: excluded {len(MULTINEWS_SKIPPED_RECORDS)} unusable "
            "record(s); see multinews_skipped_records.csv."
        )
    return rows


print("Multi-News loader ready: unusable records will be logged, not fabricated.")


In [ ]:
#@title STEP 1 — LOAD DATASET ONLY (audited denominator)
# Heavy action in this cell: Hugging Face Multi-News loading ONLY.

if "load_summarisation_data" not in globals():
    raise RuntimeError("Loader utility belum dijalankan.")

_requested_clusters = (
    min(int(CFG.max_clusters), 3)
    if getattr(CFG, "colab_smoke_test", False)
    else int(CFG.max_clusters)
)

print("=" * 78)
print("STEP 1 — MULTI-NEWS DATASET LOAD")
print("=" * 78)
print("Configured split      :", CFG.summarisation_split)
print("Requested raw records:", _requested_clusters)
if "print_ram" in globals():
    print_ram("before dataset load")

summary_data = load_summarisation_data(_requested_clusters)
if not summary_data:
    raise RuntimeError("No usable Multi-News records were loaded.")
if not all(bool(row.get("is_real_multinews", False)) for row in summary_data):
    raise RuntimeError("Non-Multi-News/demo data detected.")

cluster_ids = [row["cluster_id"] for row in summary_data]
skipped_df = pd.DataFrame(MULTINEWS_SKIPPED_RECORDS, columns=[
    "raw_record_index", "cluster_id", "reason", "dataset_source", "dataset_split"
])
save_csv(skipped_df, OUT / f"multinews_skipped_records_{MULTINEWS_SOURCE_INFO['split']}.csv", index=False)

print("Official raw split records:", MULTINEWS_SOURCE_INFO["raw_split_records"])
print("Raw records scanned       :", MULTINEWS_SOURCE_INFO["raw_records_scanned"])
print("Usable clusters loaded    :", len(summary_data))
print("Excluded unusable records :", len(skipped_df))
print("Cluster sample            :", cluster_ids[:5])

dataset_provenance_df = pd.DataFrame([{
    "dataset_name": "Multi-News",
    "dataset_source": "Awesome075/multi_news_parquet",
    "split": MULTINEWS_SOURCE_INFO["split"],
    "requested_raw_records": _requested_clusters,
    "official_raw_split_records": MULTINEWS_SOURCE_INFO["raw_split_records"],
    "raw_records_scanned": MULTINEWS_SOURCE_INFO["raw_records_scanned"],
    "usable_clusters_loaded": len(summary_data),
    "excluded_unusable_records": len(skipped_df),
    "exclusion_policy": "empty source documents or empty reference summary",
    "is_real_multinews": True,
    "fallback_demo_used": False,
    "first_cluster_id": cluster_ids[0],
    "last_cluster_id": cluster_ids[-1],
}])
save_csv(dataset_provenance_df, OUT / f"dataset_provenance_{MULTINEWS_SOURCE_INFO['split']}.csv", index=False)

if "print_ram" in globals():
    print_ram("after dataset load")
print("=" * 78)
print("STEP 1 COMPLETE — no features/scenario were run.")


## 5. Paper-1-aligned topic modelling and topic–pattern mining

This section reproduces the summarisation logic described in Paper 1: LDA topic relevance, FP-growth lexical patterns aligned with LDA topic-word weights, followed by dynamic redundancy-aware greedy sentence selection.

In [ ]:
#@title Paper-1-aligned topic modelling and topic-pattern utilities
import importlib
import subprocess
import sys
from collections import defaultdict
import time
import signal

import numpy as np
import pandas as pd
import spacy

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# FP-Growth dependency — fail fast, never auto-install inside a heavy cell.
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpgrowth
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "mlxtend belum tersedia. Install sekali pada cell terpisah "
        "(!pip install -q mlxtend), lalu rerun utility cell."
    ) from exc

try:
    # Lightweight lexical pipeline: parser/NER are unnecessary for STEP 2.
    nlp = spacy.load(
        "en_core_web_sm",
        disable=["parser", "ner"],
    )
except Exception:
    nlp = spacy.blank("en")
    print(
        "spaCy model unavailable; using English tokenizer fallback."
    )

# Explicit alias used by lexical FP-Growth preprocessing.
nlp_lex = nlp


def _normalise_rows(matrix):
    """Safe row normalisation for non-negative matrices."""
    matrix = np.asarray(matrix, dtype=float)
    if matrix.size == 0:
        return matrix
    denom = matrix.sum(axis=1, keepdims=True)
    denom[denom == 0] = 1.0
    return matrix / denom


def fit_topic_model(sentences, n_topics=20, seed=42):
    """
    Paper 1 alignment:
    - fit LDA on the candidate-sentence pool;
    - obtain sentence-topic probabilities;
    - obtain topic-word probabilities;
    - compute TS(s) by aggregating topic probabilities of words in a sentence.

    Returns a dictionary reused by topic and pattern scoring so both signals
    are derived from the SAME LDA representation.
    """
    n = len(sentences)
    if n == 0:
        return {
            "topic_score": np.array([], dtype=float),
            "theta": np.empty((0, 0)),
            "phi": np.empty((0, 0)),
            "vectorizer": None,
            "X": None,
            "feature_names": np.array([], dtype=str),
            "dominant_topic": np.array([], dtype=int),
        }

    vectorizer = CountVectorizer(
        stop_words="english",
        lowercase=True,
        max_features=5000,
        ngram_range=(1, 1),
        min_df=1,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    )

    try:
        X = vectorizer.fit_transform(sentences)
    except ValueError:
        return {
            "topic_score": np.ones(n, dtype=float),
            "theta": np.ones((n, 1), dtype=float),
            "phi": np.ones((1, 1), dtype=float),
            "vectorizer": None,
            "X": None,
            "feature_names": np.array(["fallback"], dtype=str),
            "dominant_topic": np.zeros(n, dtype=int),
        }

    # LDA cannot use more components than are meaningful for the current pool.
    effective_topics = max(
        1,
        min(int(n_topics), X.shape[0], max(1, X.shape[1]))
    )

    lda = LatentDirichletAllocation(
        n_components=effective_topics,
        random_state=seed,
        learning_method="batch",
        max_iter=10,   # explicit sklearn default
        n_jobs=1,      # stable for small per-cluster matrices
    )

    print(
        "        LDA matrix | "
        f"documents={X.shape[0]} | "
        f"features={X.shape[1]} | "
        f"effective_topics={effective_topics} | "
        "max_iter=10 | n_jobs=1",
        flush=True,
    )

    _lda_fit_start = time.perf_counter()

    theta = lda.fit_transform(X)

    print(
        "        LDA fit complete | "
        f"time={time.perf_counter() - _lda_fit_start:.2f}s",
        flush=True,
    )

    phi = _normalise_rows(lda.components_)
    feature_names = vectorizer.get_feature_names_out()

    # Paper 1: each sentence is mapped to topic relevance by aggregating
    # topic probabilities of its constituent words. We operationalise this
    # directly from LDA's topic-word probabilities.
    X_bin = (X > 0).astype(float)
    word_topic_salience = phi.max(axis=0)  # strongest topic association per word
    raw_ts = np.asarray(X_bin @ word_topic_salience).ravel()

    token_counts = np.asarray(X_bin.sum(axis=1)).ravel()
    token_counts[token_counts == 0] = 1.0
    topic_score = raw_ts / token_counts

    dominant_topic = theta.argmax(axis=1)

    return {
        "topic_score": topic_score,
        "theta": theta,
        "phi": phi,
        "vectorizer": vectorizer,
        "X": X,
        "feature_names": feature_names,
        "dominant_topic": dominant_topic,
    }


def topic_scores(sentences, n_topics=20, seed=42, return_model=False):
    model = fit_topic_model(sentences, n_topics=n_topics, seed=seed)
    if return_model:
        return model["topic_score"], model
    return model["topic_score"]


def lexical_transactions(sentences, batch_size=128):
    """
    Build lemmatised lexical transactions for FP-Growth.

    CPU optimization only:
    - same token filtering as before;
    - spaCy processes sentences with nlp.pipe(batch_size=128);
    - parser and NER remain disabled.
    """
    transactions = []

    docs = nlp_lex.pipe(
        (str(sentence).lower() for sentence in sentences),
        batch_size=int(batch_size),
    )

    for doc in docs:
        tokens = []

        for token_obj in doc:
            token = (
                token_obj.lemma_
                or token_obj.text
            ).lower().strip()

            if (
                token_obj.is_alpha
                and not token_obj.is_stop
                and len(token) > 2
            ):
                tokens.append(token)

        transactions.append(
            sorted(set(tokens))
        )

    return transactions


def _topic_word_weight_map(topic_model):
    """
    Map vocabulary terms to their strongest LDA topic-word probability.
    This is the alignment bridge between FP-growth patterns and LDA topics.
    """
    feature_names = topic_model.get("feature_names")
    phi = topic_model.get("phi")

    if feature_names is None or phi is None or len(feature_names) == 0:
        return {}

    strongest = np.asarray(phi).max(axis=0)
    return {
        str(term): float(weight)
        for term, weight in zip(feature_names, strongest)
    }



class _FPGrowthTimeout(Exception):
    pass


def _fp_timeout_handler(signum, frame):
    raise _FPGrowthTimeout(
        "FP-Growth exceeded the configured time limit."
    )


def run_fpgrowth_with_timeout(
    frame,
    min_support,
    timeout_seconds=300,
):
    """
    Run mlxtend.fpgrowth with a Linux signal timeout.
    Does not alter experimental parameters.
    """
    timeout_seconds = int(timeout_seconds)

    if timeout_seconds <= 0:
        return fpgrowth(
            frame,
            min_support=min_support,
            use_colnames=True,
            max_len=int(getattr(CFG, "max_pattern_length", 3)),
        )

    old_handler = signal.signal(
        signal.SIGALRM,
        _fp_timeout_handler,
    )

    signal.alarm(timeout_seconds)

    try:
        result = fpgrowth(
            frame,
            min_support=min_support,
            use_colnames=True,
            max_len=int(getattr(CFG, "max_pattern_length", 3)),
        )
    finally:
        signal.alarm(0)
        signal.signal(
            signal.SIGALRM,
            old_handler,
        )

    return result


def pattern_scores(
    sentences,
    topic_model,
    min_support=0.08,
):
    """
    Paper-1-aligned FP-Growth pattern scoring.

    Optimization is implementation-only:
    - batched lexical preprocessing;
    - vectorized sentence activation;
    - diagnostics/guard against pathological pattern explosion.

    min_support is never changed automatically.
    """
    if not sentences:
        return np.array([], dtype=float), []

    t_spacy = time.perf_counter()

    transactions = lexical_transactions(
        sentences,
        batch_size=128,
    )

    # Bound transaction width deterministically using the strongest LDA topic
    # probability of each lexical item. Lexical tie-breaking is stable, so all
    # platforms and scenario runs retain the same items.
    raw_widths = np.asarray([len(x) for x in transactions], dtype=float)
    max_transaction_items = max(
        1, int(getattr(CFG, "max_transaction_items", 20))
    )
    topic_item_weights = _topic_word_weight_map(topic_model)
    bounded_transactions = []
    for transaction in transactions:
        unique_transaction = sorted(set(transaction))
        ranked_items = sorted(
            unique_transaction,
            key=lambda item: (-float(topic_item_weights.get(str(item), 0.0)), str(item)),
        )
        bounded_transactions.append(ranked_items[:max_transaction_items])
    transactions = bounded_transactions

    spacy_seconds = time.perf_counter() - t_spacy

    if not any(transactions):
        globals()["_LAST_PATTERN_PROFILE"] = {
            "spacy_seconds": float(spacy_seconds),
            "fpgrowth_seconds": 0.0,
            "pattern_scoring_seconds": 0.0,
            "patterns_mined": 0,
            "transactions": int(len(transactions)),
            "unique_items": 0,
            "mean_transaction_width": 0.0,
            "max_transaction_width": 0,
            "effective_support": 0.0,
        }
        return np.zeros(len(sentences), dtype=float), []

    widths = np.asarray(
        [len(x) for x in transactions],
        dtype=float,
    )

    unique_items = len(
        set().union(
            *(set(x) for x in transactions)
        )
    )

    encoder = TransactionEncoder()
    encoded = encoder.fit(
        transactions
    ).transform(
        transactions
    )

    frame = pd.DataFrame(
        encoded,
        columns=encoder.columns_,
    )

    min_pattern_occurrences = max(
        1, int(getattr(CFG, "min_pattern_occurrences", 2))
    )
    required_occurrences = min(min_pattern_occurrences, len(transactions))
    effective_support = max(
        float(min_support),
        required_occurrences / max(1, len(transactions)),
    )

    estimated_cells = (
        int(encoded.shape[0])
        * int(encoded.shape[1])
    )

    # Diagnostic warning only; experimental support is unchanged.
    if (
        unique_items > 5000
        or float(widths.max()) > 250
        or estimated_cells > 5_000_000
    ):
        print(
            "FP-Growth preflight warning | "
            f"transactions={len(transactions)} | "
            f"unique_items={unique_items} | "
            f"mean_width={widths.mean():.1f} | "
            f"max_width={widths.max():.0f} | "
            f"support={effective_support:.4f} | min_count={required_occurrences} | "
            f"max_len={int(getattr(CFG, 'max_pattern_length', 3))} | "
            f"item_cap={max_transaction_items}"
        )

    # Safety guard prevents an indefinite hang but never changes parameters.
    if (
        unique_items > 12000
        or float(widths.max()) > 500
        or estimated_cells > 20_000_000
    ):
        raise RuntimeError(
            "FP-Growth safety guard triggered before mining. "
            f"transactions={len(transactions)}, "
            f"unique_items={unique_items}, "
            f"mean_width={widths.mean():.1f}, "
            f"max_width={widths.max():.0f}, "
            f"support={effective_support:.4f}. "
            "No experimental parameter was changed."
        )

    print(
        "    -> FP-Growth START | "
        f"transactions={len(transactions)} | "
        f"unique_items={unique_items} | "
        f"mean_width={widths.mean():.1f} | "
        f"max_width={widths.max():.0f} | "
        f"support={effective_support:.4f}",
        flush=True,
    )

    t_fp = time.perf_counter()

    try:
        freq = run_fpgrowth_with_timeout(
            frame,
            min_support=effective_support,
            timeout_seconds=getattr(
                CFG,
                "fpgrowth_timeout_seconds",
                300,
            ),
        )
    except _FPGrowthTimeout as exc:
        elapsed = time.perf_counter() - t_fp
        raise RuntimeError(
            "FP-Growth timeout on current cluster after "
            f"{elapsed:.1f}s. "
            f"transactions={len(transactions)}, "
            f"unique_items={unique_items}, "
            f"mean_width={widths.mean():.1f}, "
            f"max_width={widths.max():.0f}, "
            f"support={effective_support:.4f}. "
            "No parameter was changed; the run stopped explicitly."
        ) from exc

    fpgrowth_seconds = time.perf_counter() - t_fp

    print(
        "    -> FP-Growth DONE  | "
        f"time={fpgrowth_seconds:.2f}s | "
        f"patterns={len(freq)}",
        flush=True,
    )

    if freq.empty:
        globals()["_LAST_PATTERN_PROFILE"] = {
            "spacy_seconds": float(spacy_seconds),
            "fpgrowth_seconds": float(fpgrowth_seconds),
            "pattern_scoring_seconds": 0.0,
            "patterns_mined": 0,
            "transactions": int(len(transactions)),
            "unique_items": int(unique_items),
            "mean_transaction_width": float(widths.mean()),
            "max_transaction_width": int(widths.max()),
            "effective_support": float(effective_support),
        }
        return np.zeros(len(sentences), dtype=float), []

    # Stop explicitly if FP-Growth itself returns a pathological itemset count.
    max_frequent_itemsets = int(getattr(CFG, "max_frequent_itemsets", 500_000))
    if len(freq) > max_frequent_itemsets:
        raise RuntimeError(
            f"FP-Growth returned {len(freq):,} frequent itemsets, exceeding "
            f"the explicit cap {max_frequent_itemsets:,}. "
            f"Applied constraints: min_count={required_occurrences}, "
            f"max_len={int(getattr(CFG, 'max_pattern_length', 3))}, "
            f"item_cap={max_transaction_items}."
        )

    topic_weight = _topic_word_weight_map(
        topic_model
    )

    pattern_records = []
    scores = np.zeros(
        len(sentences),
        dtype=float,
    )

    column_index = {
        str(term): idx
        for idx, term in enumerate(
            encoder.columns_
        )
    }

    encoded_bool = np.asarray(
        encoded,
        dtype=bool,
    )

    t_score = time.perf_counter()

    # Mathematically equivalent to checking itemset subset per sentence,
    # but activation is vectorized over all sentences.
    for row in freq.itertuples(index=False):
        itemset = frozenset(
            str(x)
            for x in row.itemsets
        )

        if not itemset:
            continue

        coherence = float(
            np.mean([
                topic_weight.get(
                    term,
                    0.0,
                )
                for term in itemset
            ])
        )

        support = float(row.support)
        weight = support * coherence

        pattern_records.append({
            "items": itemset,
            "support": support,
            "topic_coherence": coherence,
            "weight": weight,
        })

        indices = [
            column_index[term]
            for term in itemset
            if term in column_index
        ]

        if not indices:
            continue

        if len(indices) == 1:
            active = encoded_bool[:, indices[0]]
        else:
            active = encoded_bool[:, indices].all(axis=1)

        scores[active] += weight

    scoring_seconds = time.perf_counter() - t_score

    globals()["_LAST_PATTERN_PROFILE"] = {
        "spacy_seconds": float(spacy_seconds),
        "fpgrowth_seconds": float(fpgrowth_seconds),
        "pattern_scoring_seconds": float(scoring_seconds),
        "patterns_mined": int(len(freq)),
        "transactions": int(len(transactions)),
        "unique_items": int(unique_items),
        "raw_mean_transaction_width": float(raw_widths.mean()) if raw_widths.size else 0.0,
        "raw_max_transaction_width": int(raw_widths.max()) if raw_widths.size else 0,
        "mean_transaction_width": float(widths.mean()),
        "max_transaction_width": int(widths.max()),
        "max_transaction_items": int(max_transaction_items),
        "min_pattern_occurrences": int(required_occurrences),
        "effective_support": float(effective_support),
    }

    return scores, pattern_records


def build_candidate_pool(topic_model, per_topic=50):
    """
    Paper 1 reports a candidate pool of top 27–50 ranked sentences per topic.
    This function uses the upper-bound setting (default 50) and takes the
    union of top sentences for every discovered LDA topic.
    """
    theta = np.asarray(topic_model.get("theta"))
    if theta.size == 0:
        return np.array([], dtype=int)

    selected = set()
    per_topic = max(1, int(per_topic))

    for topic_id in range(theta.shape[1]):
        order = np.argsort(-theta[:, topic_id])
        for idx in order[:per_topic]:
            selected.add(int(idx))

    return np.array(sorted(selected), dtype=int)


## 6. Automatic 120-scenario sensitivity analysis

Bagian ini tidak lagi memakai workflow Stage A/B/C yang terpisah. Urutan yang benar pada v2.3 adalah:

1. **Summariser utilities — DEFINE ONLY**: jalankan sekali untuk mendefinisikan fungsi pembentukan fitur dan ringkasan; sel ini memang tidak menghasilkan eksperimen.
2. **ROUGE scoring utilities**: jalankan sekali untuk mendefinisikan metrik ROUGE dan ROUGE-SU4.
3. **STEP 2 — Original triple cache**: jalankan sekali untuk membangun cache triple dari teks asli.
4. **RUN AUTOMATIC 120-SCENARIO VALIDATION GRID — RESUMABLE**: ubah hanya `RUN_ORDER_START` dan `RUN_ORDER_END` sesuai penugasan. Jangan mengubah lima parameter eksperimen secara manual karena seluruh kombinasi dibentuk otomatis dari grid pada `CFG`.
5. Jalankan sel otomatis sekali. Runner memproses seluruh run order dalam rentang tersebut, melewati skenario yang sudah lengkap, dan menggunakan kembali cache untuk `n_topics` yang sama.
6. **COMBINE ALL SAVED SINGLE-SCENARIO RESULTS + PARETO**: jalankan setelah seluruh skenario selesai.
7. **STEP 5 — BUILD REFERENCE SUMMARIES ONLY**: jalankan hanya setelah kandidat Pareto/reference dipilih untuk proses ontology/KG berikutnya.

Jadi, tidak ada kode Feature Cache, Stage A, Stage B, atau Stage C yang hilang. Fungsi satu skenario berada di scenario engine dan dipanggil otomatis oleh runner 120 skenario agar eksekusi mudah dibagi kepada beberapa mahasiswa.


Setiap skenario menyimpan precision, recall, dan F1 untuk ROUGE-1, ROUGE-2, ROUGE-L, ROUGE-LSum, dan ROUGE-SU4. Pareto menggunakan empat nilai F1 utama dan ROUGE-SU4 recall bersama preservation F1 dan compression gain.


In [ ]:
#@title Summariser utilities — DEFINE ONLY
import gc, re, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# STRICT RULE: definitions only.
# This cell does NOT load data, build feature caches, or run experiments.

import pickle

FEATURE_CACHE_DIR = (OUT / "checkpoints" / "topic_pattern_features_fpbounded_v1" / str(CFG.summarisation_split).lower())
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _feature_cache_file(cluster_id, n_topics):
    safe_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(cluster_id))
    return FEATURE_CACHE_DIR / f"{safe_id}_topics{int(n_topics)}.pkl"

def save_feature_checkpoint(cluster_id, n_topics, features):
    path = _feature_cache_file(cluster_id, n_topics)
    with open(path, "wb") as f:
        pickle.dump(features, f, protocol=pickle.HIGHEST_PROTOCOL)
    # Heavy feature files are intentionally kept local for speed.
    # Scenario result CSVs remain mirrored to Drive by save_csv().
    return path

def load_feature_checkpoint(cluster_id, n_topics):
    path = _feature_cache_file(cluster_id, n_topics)
    if not path.exists():
        return None
    with open(path, "rb") as f:
        return pickle.load(f)


required_functions = [
    "safe_sentencize",
    "topic_scores",
    "pattern_scores",
    "build_candidate_pool",
]
missing = [x for x in required_functions if x not in globals()]
if missing:
    raise RuntimeError(
        "Missing prerequisite functions: " + ", ".join(missing)
    )

def _paper1_similarity(sentences, candidate_indices=None):
    """Cosine matrix only for active candidate subset."""
    if candidate_indices is None:
        candidate_indices = list(range(len(sentences)))

    candidate_indices = list(candidate_indices)

    if not candidate_indices:
        return np.empty((0, 0)), candidate_indices

    cap = int(
        getattr(
            CFG,
            "max_sentences_for_similarity",
            220,
        )
    )

    if len(candidate_indices) > cap:
        candidate_indices = candidate_indices[:cap]

    subset = [
        sentences[i]
        for i in candidate_indices
    ]

    if len(subset) == 1:
        return (
            np.ones(
                (1, 1),
                dtype=np.float32,
            ),
            candidate_indices,
        )

    try:
        X = TfidfVectorizer(
            stop_words="english",
            lowercase=True,
            token_pattern=r"(?u)\b\w\w+\b",
        ).fit_transform(subset)

        sim = cosine_similarity(
            X,
            dense_output=True,
        ).astype(
            np.float32,
            copy=False,
        )

        del X
        return sim, candidate_indices

    except ValueError:
        return (
            np.eye(
                len(subset),
                dtype=np.float32,
            ),
            candidate_indices,
        )


# ============================================================
# FEATURE CACHE
# Stores only compact arrays, NOT LDA model objects.
# Key: (cluster_id, n_topics)
# ============================================================
SUMMARY_FEATURE_CACHE = {}


def build_summary_features(
    documents,
    n_topics,
):
    """
    Build one cluster/topic feature set and expose detailed timing.
    """
    profile = {}
    t0 = time.perf_counter()

    print("    -> sentencize START", flush=True)

    sentences = safe_sentencize(
        documents
    )

    print(
        f"    -> sentencize DONE | sentences={len(sentences)}",
        flush=True,
    )

    profile["sentence_count"] = int(len(sentences))
    profile["sentencize_seconds"] = (
        time.perf_counter() - t0
    )

    if not sentences:
        profile["total_seconds"] = time.perf_counter() - t0
        globals()["_LAST_FEATURE_PROFILE"] = profile

        return {
            "sentences": [],
            "topic": np.array([], dtype=np.float32),
            "patt": np.array([], dtype=np.float32),
            "candidates": np.array([], dtype=int),
        }

    print(
        f"    -> LDA START | n_topics={int(n_topics)}",
        flush=True,
    )

    t_lda = time.perf_counter()

    topic, model = topic_scores(
        sentences,
        n_topics=int(n_topics),
        seed=CFG.seed,
        return_model=True,
    )

    profile["lda_seconds"] = (
        time.perf_counter() - t_lda
    )

    print(
        f"    -> LDA DONE | time={profile['lda_seconds']:.2f}s",
        flush=True,
    )

    print(
        "    -> lexical + FP pattern START",
        flush=True,
    )

    t_pattern = time.perf_counter()

    patt, patterns = pattern_scores(
        sentences,
        topic_model=model,
        min_support=CFG.min_pattern_support,
    )

    profile["pattern_total_seconds"] = (
        time.perf_counter() - t_pattern
    )

    print(
        "    -> lexical + FP pattern DONE | "
        f"time={profile['pattern_total_seconds']:.2f}s",
        flush=True,
    )

    pp = globals().get(
        "_LAST_PATTERN_PROFILE",
        {},
    )

    for name, default in [
        ("spacy_seconds", 0.0),
        ("fpgrowth_seconds", 0.0),
        ("pattern_scoring_seconds", 0.0),
        ("mean_transaction_width", 0.0),
        ("effective_support", 0.0),
    ]:
        profile[name] = float(
            pp.get(name, default)
        )

    for name, default in [
        ("patterns_mined", 0),
        ("transactions", 0),
        ("unique_items", 0),
        ("max_transaction_width", 0),
    ]:
        profile[name] = int(
            pp.get(name, default)
        )

    t_pool = time.perf_counter()

    candidates = build_candidate_pool(
        model,
        per_topic=CFG.candidate_pool_per_topic,
    )

    profile["candidate_pool_seconds"] = (
        time.perf_counter() - t_pool
    )

    profile["candidate_count"] = int(
        len(candidates)
    )

    profile["total_seconds"] = (
        time.perf_counter() - t0
    )

    result = {
        "sentences": sentences,
        "topic": np.asarray(topic, dtype=np.float32),
        "patt": np.asarray(patt, dtype=np.float32),
        "candidates": np.asarray(candidates, dtype=int),
    }

    globals()["_LAST_FEATURE_PROFILE"] = profile

    del model, patterns, topic, patt

    return result


def get_cached_summary_features(
    cluster_id,
    documents,
    n_topics,
):
    key = (
        str(cluster_id),
        int(n_topics),
    )

    if key in SUMMARY_FEATURE_CACHE:
        return SUMMARY_FEATURE_CACHE[key]

    disk_features = load_feature_checkpoint(
        cluster_id,
        n_topics,
    )

    if disk_features is not None:
        SUMMARY_FEATURE_CACHE[key] = disk_features
        return disk_features

    features = build_summary_features(
        documents,
        n_topics,
    )

    SUMMARY_FEATURE_CACHE[key] = features

    save_feature_checkpoint(
        cluster_id,
        n_topics,
        features,
    )

    return features


def generate_summary_from_features(
    features,
    k=15,
    ablation="full",
    alpha=None,
    beta=None,
    delta=None,
    return_detail=False,
):
    """
    Cheap selection stage.
    Reuses cached topic and pattern scores instead of rerunning LDA/FP-growth.
    """
    sentences = features["sentences"]

    if not sentences:
        return "", (
            pd.DataFrame()
            if return_detail
            else None
        )

    target = max(
        1,
        min(
            int(k),
            len(sentences),
        ),
    )

    alpha = float(
        CFG.alpha_topic
        if alpha is None
        else alpha
    )

    beta = float(
        CFG.beta_pattern
        if beta is None
        else beta
    )

    delta = float(
        CFG.redundancy_delta
        if delta is None
        else delta
    )

    topic = features["topic"]
    patt = features["patt"]

    if ablation == "no_topic":
        topic_used = np.zeros_like(
            topic
        )
    else:
        topic_used = topic

    if ablation == "no_pattern":
        patt_used = np.zeros_like(
            patt
        )
    else:
        patt_used = patt

    candidates = features[
        "candidates"
    ]

    if (
        candidates.size == 0
        or len(candidates) < target
    ):
        candidates = np.arange(
            len(sentences),
            dtype=int,
        )

    base = (
        alpha * topic_used
        + beta * patt_used
    )

    ordered = sorted(
        (
            int(i)
            for i in candidates
        ),
        key=lambda i: float(
            base[i]
        ),
        reverse=True,
    )

    cap = max(
        target,
        int(
            getattr(
                CFG,
                "max_sentences_for_similarity",
                220,
            )
        ),
    )

    ordered = ordered[:cap]

    sim, active = _paper1_similarity(
        sentences,
        ordered,
    )

    pos = {
        idx: j
        for j, idx
        in enumerate(active)
    }

    selected = []
    remaining = set(active)
    selected_scores = {}

    while (
        remaining
        and len(selected) < target
    ):
        best_idx = None
        best_score = -np.inf
        best_red = 0.0

        for idx in remaining:
            if (
                selected
                and ablation
                != "no_redundancy"
            ):
                pi = pos[idx]
                red = max(
                    float(
                        sim[
                            pi,
                            pos[j],
                        ]
                    )
                    for j in selected
                )
            else:
                red = 0.0

            score = (
                alpha
                * float(
                    topic_used[idx]
                )
                + beta
                * float(
                    patt_used[idx]
                )
                - delta * red
            )

            if score > best_score:
                best_idx = idx
                best_score = score
                best_red = red

        if best_idx is None:
            break

        selected.append(
            best_idx
        )
        remaining.remove(
            best_idx
        )
        selected_scores[
            best_idx
        ] = (
            best_score,
            best_red,
        )

    summary = " ".join(
        sentences[i]
        for i in sorted(selected)
    )

    detail = None

    if return_detail:
        detail = pd.DataFrame({
            "sentence_id":
                selected,

            "sentence":
                [
                    sentences[i]
                    for i in selected
                ],

            "topic_score_TS":
                [
                    float(
                        topic_used[i]
                    )
                    for i in selected
                ],

            "pattern_score_PR":
                [
                    float(
                        patt_used[i]
                    )
                    for i in selected
                ],

            "final_score":
                [
                    selected_scores[i][0]
                    for i in selected
                ],

            "redundancy":
                [
                    selected_scores[i][1]
                    for i in selected
                ],
        })

    del sim, base

    return summary, detail


def generate_summary(
    documents,
    k=15,
    ablation="full",
    alpha=None,
    beta=None,
    delta=None,
    n_topics=None,
    return_detail=False,
):
    """
    Backward-compatible wrapper for cells outside Pareto tuning.
    It builds features once for this call.
    """
    n_topics = int(
        CFG.n_topics
        if n_topics is None
        else n_topics
    )

    # Exact fast path for extractive summaries: when the requested budget is
    # at least the number of available sentences, selecting every sentence is
    # the only valid result. LDA/FP-Growth would not change that result.
    sentences = safe_sentencize(documents)
    if len(sentences) <= int(k):
        full_text = " ".join(sentences)
        detail = None
        if return_detail:
            detail = pd.DataFrame({
                "sentence": sentences,
                "selected": [True] * len(sentences),
                "selection_order": list(range(1, len(sentences) + 1)),
            })
        return full_text, detail

    features = build_summary_features(
        documents,
        n_topics,
    )

    return generate_summary_from_features(
        features,
        k=k,
        ablation=ablation,
        alpha=alpha,
        beta=beta,
        delta=delta,
        return_detail=return_detail,
    )


print(
    "Summariser utilities ready. No data/features/scenario were run."
)
print_ram(
    "after summariser setup"
)


## 7. ROUGE evaluation

In [ ]:
#@title ROUGE scoring utilities — ROUGE-1/2/L/LSum + ROUGE-SU4
import importlib
import subprocess
import sys
import re
from collections import Counter

import numpy as np
import pandas as pd

try:
    from rouge_score import rouge_scorer
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "rouge-score==0.1.2"
    ])
    importlib.invalidate_caches()
    from rouge_score import rouge_scorer


def _rouge_tokens(text):
    return re.findall(r"[A-Za-z0-9]+", str(text).lower())


def _skip_bigrams(tokens, max_skip=4):
    result = []
    n = len(tokens)
    for i in range(n):
        max_j = min(n, i + max_skip + 2)
        for j in range(i + 1, max_j):
            result.append((tokens[i], tokens[j]))
    return result


def _multiset_overlap(reference_items, prediction_items):
    return sum((Counter(reference_items) & Counter(prediction_items)).values())


def rouge_su4(reference, prediction, max_skip=4):
    ref_tokens = _rouge_tokens(reference)
    pred_tokens = _rouge_tokens(prediction)
    ref_units = [("U", t) for t in ref_tokens]
    pred_units = [("U", t) for t in pred_tokens]
    ref_units += [("S", a, b) for a, b in _skip_bigrams(ref_tokens, max_skip)]
    pred_units += [("S", a, b) for a, b in _skip_bigrams(pred_tokens, max_skip)]
    overlap = _multiset_overlap(ref_units, pred_units)
    precision = overlap / len(pred_units) if pred_units else 0.0
    recall = overlap / len(ref_units) if ref_units else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def _prepare_rouge_lsum(text):
    """Return newline-separated sentences required by rouge-score ROUGE-LSum."""
    text = str(text or "").strip()
    if not text:
        return ""
    # Preserve existing sentence newlines and add boundaries for ordinary text.
    chunks = []
    for paragraph in text.splitlines() or [text]:
        chunks.extend(re.split(r"(?<=[.!?])\s+", paragraph.strip()))
    return "\n".join(chunk.strip() for chunk in chunks if chunk.strip())


rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)
rouge_lsum = rouge_scorer.RougeScorer(["rougeLsum"], use_stemmer=True)


def compute_all_rouge(reference, prediction):
    reference = str(reference or "")
    prediction = str(prediction or "")
    standard = rouge.score(reference, prediction)
    lsum = rouge_lsum.score(
        _prepare_rouge_lsum(reference), _prepare_rouge_lsum(prediction)
    )["rougeLsum"]
    su4_p, su4_r, su4_f = rouge_su4(reference, prediction, max_skip=4)
    return {
        "rouge1_precision": float(standard["rouge1"].precision),
        "rouge1_recall": float(standard["rouge1"].recall),
        "rouge1_f1": float(standard["rouge1"].fmeasure),
        "rouge2_precision": float(standard["rouge2"].precision),
        "rouge2_recall": float(standard["rouge2"].recall),
        "rouge2_f1": float(standard["rouge2"].fmeasure),
        "rougeL_precision": float(standard["rougeL"].precision),
        "rougeL_recall": float(standard["rougeL"].recall),
        "rougeL_f1": float(standard["rougeL"].fmeasure),
        "rougeLsum_precision": float(lsum.precision),
        "rougeLsum_recall": float(lsum.recall),
        "rougeLsum_f1": float(lsum.fmeasure),
        "rougeSU4_precision": float(su4_p),
        "rougeSU4_recall": float(su4_r),
        "rougeSU4_f1": float(su4_f),
    }


def score_summary_frame(frame):
    rows = []
    for row in frame.itertuples(index=False):
        reference = str(getattr(row, "reference_summary", "") or "")
        prediction = str(getattr(row, "predicted_summary", "") or "")
        rows.append({
            "cluster_id": getattr(row, "cluster_id"),
            "configuration_id": getattr(row, "configuration_id", ""),
            "alpha_topic": float(getattr(row, "alpha_topic")),
            "beta_pattern": float(getattr(row, "beta_pattern")),
            "delta_redundancy": float(getattr(row, "delta_redundancy")),
            "n_topics": int(getattr(row, "n_topics")),
            "summary_sentence_budget": int(getattr(row, "summary_sentence_budget")),
            **compute_all_rouge(reference, prediction),
        })
    return pd.DataFrame(rows)


print("ROUGE utilities ready: ROUGE-1/2/L/LSum P/R/F1 and ROUGE-SU4 P/R/F1.")


## 8. Triple extraction and semantic-preservation comparison

This section extracts triples from **both the original Multi-News text and its Paper-1-based summary**. The original-text triples are used only as a within-pipeline reference for measuring knowledge preservation after compression; they are **not treated as gold-standard triples**. Independent correctness of the extractor is evaluated later with WebNLG.

**v7 speed optimisation.** LDA/topic scores and FP-growth pattern scores are computed once per Multi-News cluster and `n_topics`, then reused across all α–β–δ and 10/15 configurations. Summary triple parsing is also cached by generated summary text.

In [ ]:
#@title STEP 2 — BUILD ORIGINAL TRIPLE CACHE WITH BATCHED spaCy
# Heavy action here: dependency parsing of ORIGINAL Multi-News text only.
# Uses nlp.pipe() and per-cluster checkpointing.

import gc
import json
import re
import time
import spacy

if "summary_data" not in globals() or not summary_data:
    raise RuntimeError(
        "summary_data belum tersedia. Jalankan STEP 1."
    )

try:
    nlp_dep = spacy.load(
        "en_core_web_sm",
        disable=["ner"],
    )
except OSError as exc:
    raise RuntimeError(
        "spaCy model en_core_web_sm belum tersedia. "
        "Install sekali dengan: !python -m spacy download en_core_web_sm"
    ) from exc

TRIPLE_CACHE_DIR = (OUT / "checkpoints" / "original_triples" / str(CFG.summarisation_split).lower())
TRIPLE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def clean_text(v):
    return " ".join(
        str(v or "").replace("\u00a0", " ").split()
    )

def normalize_label(v):
    return re.sub(
        r"[^a-z0-9]+", "_", clean_text(v).lower()
    ).strip("_")

def subtree_text(tok):
    return clean_text(
        " ".join(
            x.text
            for x in sorted(tok.subtree, key=lambda z: z.i)
        )
    )

def extract_dependency_triples_from_doc(doc):
    out = []
    seen = set()

    for sent in doc.sents:
        roots = [
            t for t in sent
            if t.dep_ == "ROOT"
            and t.pos_ in {"VERB", "AUX"}
        ]

        for root in roots:
            children = list(root.children)

            subjects = [
                t for t in children
                if t.dep_ in {"nsubj", "nsubjpass", "csubj"}
            ]

            objects = [
                t for t in children
                if t.dep_ in {"dobj", "obj", "attr", "oprd", "dative"}
            ]

            candidates = []

            for sub in subjects:
                for obj in objects:
                    candidates.append(
                        (
                            subtree_text(sub),
                            root.lemma_ or root.text,
                            subtree_text(obj),
                        )
                    )

            for prep in [t for t in children if t.dep_ == "prep"]:
                prep_objects = [
                    t for t in prep.children
                    if t.dep_ in {"pobj", "obj"}
                ]
                for sub in subjects:
                    for obj in prep_objects:
                        candidates.append(
                            (
                                subtree_text(sub),
                                f"{root.lemma_ or root.text}_{prep.lemma_ or prep.text}",
                                subtree_text(obj),
                            )
                        )

            for sub, pred, obj in candidates:
                key = (
                    normalize_label(sub),
                    normalize_label(pred),
                    normalize_label(obj),
                )
                if all(key) and key not in seen:
                    seen.add(key)
                    out.append({
                        "subject": sub,
                        "predicate": pred,
                        "object": obj,
                    })

    return out

def extract_dependency_triples(text_value):
    txt = clean_text(text_value)
    if not txt:
        return []
    return extract_dependency_triples_from_doc(
        nlp_dep(txt)
    )

def _canon(t):
    vals = (
        (
            t.get("subject", ""),
            t.get("predicate", ""),
            t.get("object", ""),
        )
        if isinstance(t, dict)
        else t[:3]
    )
    return tuple(normalize_label(x) for x in vals)

def _rel(v):
    return "".join(
        c for c in normalize_label(v) if c.isalnum()
    )

def preservation_soft_match(a, b):
    ps, pp, po = _canon(a)
    rs, rp, ro = _canon(b)
    pp = _rel(pp)
    rp = _rel(rp)
    return bool(
        all([ps, pp, po, rs, rp, ro])
        and (ps in rs or rs in ps)
        and (po in ro or ro in po)
        and (pp in rp or rp in pp)
    )

def preservation_prf(pred, ref):
    used = set()
    tp = 0

    for p in pred:
        for j, r in enumerate(ref):
            if j not in used and preservation_soft_match(p, r):
                used.add(j)
                tp += 1
                break

    fp = len(pred) - tp
    fn = len(ref) - tp
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return P, R, F

def triple_checkpoint_path(cluster_id):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(cluster_id))
    return TRIPLE_CACHE_DIR / f"{safe}.json"

original_cache = {}
pending_rows = []

for row in summary_data:
    path = triple_checkpoint_path(row["cluster_id"])

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            original_cache[row["cluster_id"]] = json.load(f)
    else:
        pending_rows.append(row)

print("=" * 78)
print("STEP 2B — BATCHED ORIGINAL TRIPLE CACHE")
print("=" * 78)
print("Already restored:", len(original_cache))
print("Need parsing    :", len(pending_rows))

start_time = time.time()

if pending_rows:
    texts = [
        clean_text(" ".join(row["documents"]))
        for row in pending_rows
    ]

    docs = nlp_dep.pipe(
        texts,
        batch_size=4,
    )

    for idx, (row, txt, doc) in enumerate(
        zip(pending_rows, texts, docs),
        start=1,
    ):
        record = {
            "words": len(txt.split()),
            "triples": extract_dependency_triples_from_doc(doc),
        }

        original_cache[row["cluster_id"]] = record

        path = triple_checkpoint_path(row["cluster_id"])
        with open(path, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False)

        if idx % 5 == 0 or idx == len(pending_rows):
            elapsed = time.time() - start_time
            print(
                f"parsed {idx}/{len(pending_rows)} pending clusters | "
                f"elapsed={elapsed/60:.1f} min"
            )
            gc.collect()

    del texts, docs

SUMMARY_TRIPLE_CACHE = {}

elapsed = time.time() - start_time
print("Original triple-cache entries:", len(original_cache))
print(f"Elapsed this run: {elapsed/60:.2f} minutes")
print("=" * 78)
print("STEP 2B COMPLETE — no scenario was evaluated.")


In [ ]:
#@title Scenario engine — DEFINE ONLY (manual + automatic runner use this)
import gc
import time
import traceback
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

for required_name in [
    "summary_data", "get_cached_summary_features", "original_cache",
    "compute_all_rouge", "extract_dependency_triples", "preservation_prf",
]:
    if required_name not in globals():
        raise RuntimeError(f"Missing prerequisite: {required_name}. Run preceding cells in order.")


def active_experiment_split():
    split = str(getattr(CFG, "summarisation_split", "validation")).lower().strip()
    if split not in {"train", "validation", "test"}:
        raise ValueError(f"Unsupported split: {split}")
    return split


def scenario_result_file():
    return OUT / f"single_scenario_results_{active_experiment_split()}.csv"


def scenario_detail_dir():
    path = OUT / f"scenario_details_{active_experiment_split()}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def make_scenario_id(alpha, beta, delta, budget, n_topics):
    return (
        f"{active_experiment_split()}_a{float(alpha):.2f}_b{float(beta):.2f}_"
        f"d{float(delta):.2f}_len{int(budget)}_topics{int(n_topics)}_"
        f"clusters{len(summary_data)}"
    )


_ACTIVE_FEATURE_TOPIC = None
_ACTIVE_FEATURES = {}


def prepare_topic_feature_cache(n_topics, verbose=True):
    """Build/load features once for every cluster at one topic setting."""
    global _ACTIVE_FEATURE_TOPIC, _ACTIVE_FEATURES, SUMMARY_FEATURE_CACHE
    n_topics = int(n_topics)
    if _ACTIVE_FEATURE_TOPIC == n_topics and len(_ACTIVE_FEATURES) == len(summary_data):
        return _ACTIVE_FEATURES

    # Release the previous topic representation before loading the next one.
    _ACTIVE_FEATURES = {}
    if "SUMMARY_FEATURE_CACHE" in globals():
        SUMMARY_FEATURE_CACHE.clear()
    cleanup_memory()

    profile_rows = []
    profile_file = OUT / f"feature_profile_{active_experiment_split()}_topics{n_topics}.csv"
    started_all = time.perf_counter()
    for idx, row in enumerate(summary_data, start=1):
        cluster_id = str(row["cluster_id"])
        cache_path = _feature_cache_file(cluster_id, n_topics)
        was_cached = cache_path.exists()
        started = time.perf_counter()
        features = get_cached_summary_features(cluster_id, row["documents"], n_topics)
        _ACTIVE_FEATURES[cluster_id] = features
        elapsed = time.perf_counter() - started
        profile = dict(globals().get("_LAST_FEATURE_PROFILE", {})) if not was_cached else {
            "sentence_count": len(features.get("sentences", [])),
            "lda_seconds": 0.0, "spacy_seconds": 0.0,
            "fpgrowth_seconds": 0.0, "pattern_scoring_seconds": 0.0,
            "patterns_mined": len(features.get("patterns", [])),
            "total_seconds": elapsed,
        }
        profile["cache_status"] = "reused" if was_cached else "built"
        profile_rows.append({"cluster_id": cluster_id, "n_topics": n_topics, **profile})
        if verbose and (idx == 1 or idx % 100 == 0 or idx == len(summary_data)):
            print(
                f"feature cache topics={n_topics}: {idx}/{len(summary_data)} | "
                f"{profile['cache_status']} | {elapsed:.2f}s",
                flush=True,
            )
        if idx % 250 == 0:
            save_csv(pd.DataFrame(profile_rows), profile_file, index=False)
            cleanup_memory()
    save_csv(pd.DataFrame(profile_rows), profile_file, index=False)
    _ACTIVE_FEATURE_TOPIC = n_topics
    print(
        f"Feature cache ready | topics={n_topics} | clusters={len(_ACTIVE_FEATURES)} | "
        f"minutes={(time.perf_counter()-started_all)/60:.2f}"
    )
    return _ACTIVE_FEATURES


def _append_exception(scenario_id, cluster_id, exc):
    path = OUT / f"scenario_exceptions_{active_experiment_split()}.csv"
    row = pd.DataFrame([{
        "scenario_id": scenario_id,
        "cluster_id": cluster_id,
        "exception_type": type(exc).__name__,
        "message": str(exc),
        "traceback": traceback.format_exc(limit=8),
    }])
    if path.exists():
        old = pd.read_csv(path, keep_default_na=False)
        row = pd.concat([old, row], ignore_index=True)
    save_csv(row, path, index=False)


def run_one_complete_scenario(alpha, beta, delta, budget, n_topics, run_order=None, verbose=True):
    """Run or resume exactly one configuration and append one aggregate row."""
    alpha, beta, delta = float(alpha), float(beta), float(delta)
    budget, n_topics = int(budget), int(n_topics)
    if alpha < 0 or beta < 0 or not np.isclose(alpha + beta, 1.0):
        raise ValueError("alpha and beta must be non-negative and sum to 1.0")
    if delta < 0 or budget < 1 or n_topics < 2:
        raise ValueError("Invalid delta, summary length, or n_topics")

    scenario_id = make_scenario_id(alpha, beta, delta, budget, n_topics)
    result_file = scenario_result_file()
    detail_file = scenario_detail_dir() / f"scenario_details_{scenario_id}.csv"
    existing_results = pd.read_csv(result_file) if result_file.exists() else pd.DataFrame()
    is_complete = (
        not existing_results.empty
        and "configuration_id" in existing_results.columns
        and scenario_id in set(existing_results["configuration_id"].astype(str))
        and set(CFG.saved_metrics).issubset(existing_results.columns)
    )
    if is_complete:
        print(f"[SKIP COMPLETE] run_order={run_order} | {scenario_id}")
        return existing_results[existing_results["configuration_id"].astype(str) == scenario_id].iloc[-1].to_dict()

    features_by_cluster = prepare_topic_feature_cache(n_topics, verbose=verbose)
    aggregate_metric_names = list(dict.fromkeys([
        *CFG.saved_metrics,
        "rougeSU4_precision", "rougeSU4_f1",
        "preservation_precision", "preservation_recall", "compression_ratio",
    ]))

    if detail_file.exists():
        detail_df = pd.read_csv(detail_file, keep_default_na=False)
        # A partial checkpoint is reusable only for this exact configuration.
        if not detail_df.empty and set(detail_df.get("configuration_id", [])) != {scenario_id}:
            raise RuntimeError(f"Checkpoint configuration mismatch: {detail_file.name}")
        detail_rows = detail_df.to_dict("records")
        completed_ids = set(detail_df.get("cluster_id", pd.Series(dtype=str)).astype(str))
    else:
        detail_rows, completed_ids = [], set()

    print("=" * 88)
    print(f"RUN ORDER {run_order} | {scenario_id} | resume={len(completed_ids)}/{len(summary_data)}")
    print("=" * 88)
    scenario_start = time.perf_counter()
    summary_triple_cache = {}
    checkpoint_every = max(1, int(getattr(CFG, "grid_checkpoint_every", 250)))

    for idx, row in enumerate(summary_data, start=1):
        cluster_id = str(row["cluster_id"])
        if cluster_id in completed_ids:
            continue
        try:
            pred, _ = generate_summary_from_features(
                features_by_cluster[cluster_id], k=budget,
                alpha=alpha, beta=beta, delta=delta, return_detail=False,
            )
            rouge_values = compute_all_rouge(row["reference_summary"], pred)
            if pred not in summary_triple_cache:
                if len(summary_triple_cache) >= int(getattr(CFG, "grid_triple_cache_limit", 5000)):
                    summary_triple_cache.clear()
                summary_triple_cache[pred] = extract_dependency_triples(pred)
            p, r, f1 = preservation_prf(
                summary_triple_cache[pred], original_cache[cluster_id]["triples"]
            )
            original_words = int(original_cache[cluster_id]["words"])
            summary_words = len(pred.split())
            ratio = summary_words / original_words if original_words else 0.0
            values = {
                **rouge_values,
                "preservation_precision": float(p),
                "preservation_recall": float(r),
                "preservation_f1": float(f1),
                "compression_ratio": float(ratio),
                "compression_gain": float(1.0 - ratio),
            }
            detail = {
                "run_order": run_order,
                "configuration_id": scenario_id,
                "cluster_id": cluster_id,
                "alpha_topic": alpha,
                "beta_pattern": beta,
                "delta_redundancy": delta,
                "n_topics": n_topics,
                "summary_sentence_budget": budget,
                **values,
            }
            if bool(getattr(CFG, "grid_save_summary_text", False)):
                detail["reference_summary"] = row["reference_summary"]
                detail["generated_summary"] = pred
            detail_rows.append(detail)
        except Exception as exc:
            save_csv(pd.DataFrame(detail_rows), detail_file, index=False)
            _append_exception(scenario_id, cluster_id, exc)
            raise RuntimeError(
                f"Scenario failed at cluster {cluster_id}; partial checkpoint saved."
            ) from exc

        processed = len(detail_rows)
        if processed % checkpoint_every == 0 or processed == len(summary_data):
            save_csv(pd.DataFrame(detail_rows), detail_file, index=False)
            if verbose:
                print(f"scenario {run_order}: {processed}/{len(summary_data)}", flush=True)
            cleanup_memory()

    detail_df = pd.DataFrame(detail_rows)
    if len(detail_df) != len(summary_data) or detail_df["cluster_id"].nunique() != len(summary_data):
        raise RuntimeError(
            f"Incomplete scenario {scenario_id}: rows={len(detail_df)}, "
            f"unique_clusters={detail_df['cluster_id'].nunique()}, expected={len(summary_data)}"
        )
    if detail_df[aggregate_metric_names].isna().any().any():
        raise RuntimeError(f"NaN metric detected in {scenario_id}")

    result = {
        "run_order": run_order,
        "dataset_split": active_experiment_split(),
        "configuration_id": scenario_id,
        "alpha_topic": alpha,
        "beta_pattern": beta,
        "delta_redundancy": delta,
        "n_topics": n_topics,
        "summary_sentence_budget": budget,
        "n_clusters": len(detail_df),
        **{name: float(detail_df[name].mean()) for name in aggregate_metric_names},
        "runtime_minutes": (time.perf_counter() - scenario_start) / 60.0,
        "status": "complete",
    }
    updated = pd.concat([existing_results, pd.DataFrame([result])], ignore_index=True)
    updated = updated.drop_duplicates("configuration_id", keep="last")
    save_csv(updated.sort_values("run_order", kind="stable"), result_file, index=False)
    print(f"[COMPLETE] run_order={run_order} | {scenario_id}")
    return result


print("Scenario engine ready. No scenario was run in this definition cell.")


In [ ]:
#@title RUN AUTOMATIC 120-SCENARIO VALIDATION GRID — RESUMABLE
from itertools import product
import pandas as pd

# For one-machine execution leave 1..120. For four students use 1..30,
# 31..60, 61..90, and 91..120 respectively.
RUN_ORDER_START = 1
RUN_ORDER_END = 120

if active_experiment_split() != "validation":
    raise RuntimeError(
        "Sensitivity tuning is permitted only on the validation split. "
        "Set CFG.summarisation_split='validation', rerun the loader and caches, then retry."
    )
if bool(getattr(CFG, "colab_smoke_test", False)):
    raise RuntimeError("Disable colab_smoke_test before the definitive 120-scenario run.")

# Topic is outermost so the expensive LDA/FP feature cache is reused for all
# 40 configurations at the same n_topics value.
scenario_manifest = []
run_order = 0
for n_topics, (alpha, beta), delta, budget in product(
    CFG.n_topics_grid,
    CFG.alpha_beta_grid,
    CFG.delta_grid,
    CFG.summary_length_grid,
):
    run_order += 1
    scenario_manifest.append({
        "run_order": run_order,
        "alpha_topic": float(alpha),
        "beta_pattern": float(beta),
        "delta_redundancy": float(delta),
        "summary_sentence_budget": int(budget),
        "n_topics": int(n_topics),
        "configuration_id": make_scenario_id(alpha, beta, delta, budget, n_topics),
    })

manifest_df = pd.DataFrame(scenario_manifest)
if len(manifest_df) != 120 or manifest_df["configuration_id"].nunique() != 120:
    raise RuntimeError("The sensitivity grid must contain exactly 120 unique configurations.")
save_csv(manifest_df, OUT / "paper2_120_scenario_manifest.csv", index=False)

if not (1 <= int(RUN_ORDER_START) <= int(RUN_ORDER_END) <= 120):
    raise ValueError("RUN_ORDER_START/END must satisfy 1 <= START <= END <= 120")

selected_manifest = manifest_df[
    manifest_df["run_order"].between(int(RUN_ORDER_START), int(RUN_ORDER_END))
].copy()
print(f"Running {len(selected_manifest)} scenarios: {RUN_ORDER_START}..{RUN_ORDER_END}")
print("Result file:", scenario_result_file())
display(selected_manifest)

for row in selected_manifest.itertuples(index=False):
    run_one_complete_scenario(
        alpha=row.alpha_topic,
        beta=row.beta_pattern,
        delta=row.delta_redundancy,
        budget=row.summary_sentence_budget,
        n_topics=row.n_topics,
        run_order=row.run_order,
        verbose=True,
    )

ALL_120_RESULTS = pd.read_csv(scenario_result_file())
completed_orders = set(pd.to_numeric(ALL_120_RESULTS["run_order"], errors="coerce").dropna().astype(int))
requested_orders = set(selected_manifest["run_order"].astype(int))
missing_orders = sorted(requested_orders - completed_orders)
if missing_orders:
    raise RuntimeError(f"Requested run orders are incomplete: {missing_orders}")

print("Selected range complete:", RUN_ORDER_START, "..", RUN_ORDER_END)
print("Total accumulated scenarios in file:", len(ALL_120_RESULTS))
display(ALL_120_RESULTS.sort_values("run_order", kind="stable"))


In [ ]:
#@title COMBINE ALL SAVED SINGLE-SCENARIO RESULTS + PARETO
import numpy as np
import pandas as pd
from IPython.display import display

result_file = scenario_result_file()
if not result_file.exists():
    raise RuntimeError("Belum ada hasil. Jalankan cell ONE COMPLETE SCENARIO minimal sekali.")

all_scenarios = pd.read_csv(result_file)
if active_experiment_split() == "validation" and all_scenarios["configuration_id"].nunique() != 120:
    raise RuntimeError(
        f"Pareto aggregation requires 120 unique validation scenarios; "
        f"found {all_scenarios['configuration_id'].nunique()}."
    )
missing_saved = sorted(set(CFG.saved_metrics) - set(all_scenarios.columns))
if missing_saved:
    raise RuntimeError(
        "Saved scenario file is stale/incomplete. Missing metrics: "
        + ", ".join(missing_saved)
        + ". Rerun the affected scenarios with the updated ROUGE cell."
    )
metrics = list(CFG.pareto_metrics)
if all_scenarios[metrics].isna().any().any():
    raise RuntimeError("Pareto metrics contain missing values; inspect scenario outputs.")

def pareto_front(df, columns):
    values = df[columns].fillna(-np.inf).to_numpy(float)
    keep = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        for j in range(len(df)):
            if i != j and np.all(values[j] >= values[i]) and np.any(values[j] > values[i]):
                keep[i] = False
                break
    out = df.loc[keep].copy().reset_index(drop=True)
    out["pareto_optimal"] = True
    return out

FINAL_PARETO_CONFIGS = pareto_front(all_scenarios, metrics)
save_csv(FINAL_PARETO_CONFIGS, OUT / f"final_pareto_configurations_{active_experiment_split()}.csv", index=False)
print("Saved scenarios:", len(all_scenarios))
print("Pareto configurations:", len(FINAL_PARETO_CONFIGS))
display(all_scenarios.sort_values(["n_topics", "summary_sentence_budget"]))
display(FINAL_PARETO_CONFIGS)

# Select exactly one reproducible downstream reference from the Pareto front.
# This is a lexicographic tie-break, not a weighted composite objective:
# preservation F1 -> ROUGE-2 -> ROUGE-SU4 -> compression gain -> runtime.
if FINAL_PARETO_CONFIGS.empty:
    raise RuntimeError("Pareto front is empty; inspect scenario metric columns.")

priority = [
    c for c in [
        "preservation_f1",
        "rouge2_f1",
        "rougeSU4_recall",
        "compression_gain",
    ]
    if c in FINAL_PARETO_CONFIGS.columns
]
ascending = [False] * len(priority)
if "runtime_minutes" in FINAL_PARETO_CONFIGS.columns:
    priority.append("runtime_minutes")
    ascending.append(True)

ranked_pareto = FINAL_PARETO_CONFIGS.sort_values(
    priority,
    ascending=ascending,
    kind="stable",
).reset_index(drop=True)

REFERENCE_CONFIG = ranked_pareto.iloc[0]
REFERENCE_ALPHA = float(REFERENCE_CONFIG["alpha_topic"])
REFERENCE_BETA = float(REFERENCE_CONFIG["beta_pattern"])
REFERENCE_DELTA = float(REFERENCE_CONFIG["delta_redundancy"])
REFERENCE_SUMMARY_LENGTH = int(REFERENCE_CONFIG["summary_sentence_budget"])
REFERENCE_N_TOPICS = int(REFERENCE_CONFIG["n_topics"])

reference_configuration_df = ranked_pareto.iloc[[0]].copy()
save_csv(
    reference_configuration_df,
    OUT / f"selected_reference_configuration_{active_experiment_split()}.csv",
    index=False,
)

print("\nSelected downstream reference configuration:")
display(reference_configuration_df)
print(
    "REFERENCE_* ready:",
    REFERENCE_ALPHA,
    REFERENCE_BETA,
    REFERENCE_DELTA,
    REFERENCE_SUMMARY_LENGTH,
    REFERENCE_N_TOPICS,
)


In [ ]:
#@title STEP 5 — BUILD REFERENCE SUMMARIES ONLY
# Needed only before RDF/KG and downstream comparison.
# Uses cached topic/pattern features; does not retune parameters.

import pandas as pd

required = [
    "REFERENCE_ALPHA",
    "REFERENCE_BETA",
    "REFERENCE_DELTA",
    "REFERENCE_SUMMARY_LENGTH",
    "REFERENCE_N_TOPICS",
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run COMBINE ALL SAVED SINGLE-SCENARIO RESULTS + PARETO first. Missing: " + ", ".join(missing)
    )

if "summary_data" not in globals() or not summary_data:
    raise RuntimeError(
        "Run STEP 1 first."
    )

# Feature files may have been produced by another student/runtime.
# get_cached_summary_features restores a local cache when available and
# otherwise rebuilds the selected reference features automatically.
print(
    "Building/restoring reference features for n_topics=",
    REFERENCE_N_TOPICS,
)

reference_rows = []

for idx, row in enumerate(summary_data, start=1):
    features = get_cached_summary_features(
        row["cluster_id"],
        row["documents"],
        REFERENCE_N_TOPICS,
    )

    pred, _ = generate_summary_from_features(
        features,
        k=REFERENCE_SUMMARY_LENGTH,
        alpha=REFERENCE_ALPHA,
        beta=REFERENCE_BETA,
        delta=REFERENCE_DELTA,
        return_detail=False,
    )

    reference_rows.append({
        "cluster_id": row["cluster_id"],
        "predicted_summary": pred,
        "reference_summary": row["reference_summary"],
    })

    if idx % 5 == 0 or idx == len(summary_data):
        print(
            f"reference summary progress: {idx}/{len(summary_data)}"
        )

final_selected_summary_df = pd.DataFrame(reference_rows)
summary_df = final_selected_summary_df.copy()
pred_summaries = summary_df.to_dict("records")

save_csv(
    final_selected_summary_df,
    OUT / "final_selected_reference_summaries.csv",
    index=False,
)

print(
    "STEP 5 complete:",
    len(final_selected_summary_df),
    "reference summaries saved."
)


## 9. Optional REBEL neural relation extraction baseline (GPU recommended)

In [ ]:
#@title Optional REBEL relation-extraction baseline
REBEL_PIPE = None
use_rebel = getattr(CFG, "use_rebel", False) if "CFG" in globals() else False
rebel_model = (
    getattr(CFG, "rebel_model", "Babelscape/rebel-large")
    if "CFG" in globals() else "Babelscape/rebel-large"
)
device_name = globals().get("DEVICE", "cpu")

if use_rebel:
    try:
        from transformers import pipeline
        print(f"Loading optional REBEL model: {rebel_model}")
        REBEL_PIPE = pipeline(
            "text2text-generation",
            model=rebel_model,
            tokenizer=rebel_model,
            device=0 if device_name == "cuda" else -1,
        )
    except Exception as exc:
        print("REBEL baseline unavailable; continuing without it.")
        print("Detail:", repr(exc))
        REBEL_PIPE = None
else:
    print("REBEL baseline disabled (default).")

def parse_rebel_output(text):
    triples = []
    subject = relation = object_ = None
    state = None
    cleaned = str(text).replace("<s>", "").replace("</s>", "").replace("<pad>", "")
    for token in cleaned.split():
        if token == "<triplet>":
            if subject and relation and object_:
                triples.append((subject.strip(), relation.strip(), object_.strip()))
            subject, relation, object_, state = "", "", "", "subject"
        elif token == "<subj>":
            state = "object"
        elif token == "<obj>":
            state = "relation"
        elif state == "subject":
            subject += " " + token
        elif state == "object":
            object_ += " " + token
        elif state == "relation":
            relation += " " + token
    if subject and relation and object_:
        triples.append((subject.strip(), relation.strip(), object_.strip()))
    return triples

def extract_rebel_triples(text):
    if REBEL_PIPE is None:
        return []
    generated = REBEL_PIPE(
        str(text)[:3000],
        max_length=512,
        num_beams=3,
        return_tensors=False,
    )[0]["generated_text"]
    return [
        {
            "subject": subject,
            "predicate": predicate,
            "object": obj,
            "evidence": str(text),
            "method": "rebel",
        }
        for subject, predicate, obj in parse_rebel_output(generated)
    ]


## 10. RDF/OWL knowledge graph construction and export

In [ ]:
# ================================================================
# Provenance guard for ontology / KG construction
# ================================================================
if str(CFG.run_mode).upper() == "FULL":
    if not MULTINEWS_SOURCE_INFO.get("is_real_multinews", False):
        raise RuntimeError(
            "Ontology/KG construction aborted: "
            "source is not real Multi-News."
        )

print(
    "Ontology/KG source:",
    MULTINEWS_SOURCE_INFO.get("dataset_name"),
    "|",
    MULTINEWS_SOURCE_INFO.get("dataset_source"),
    "| split=",
    MULTINEWS_SOURCE_INFO.get("split"),
)

# Build compact final triple tables from reference summaries already produced in low-memory Stage C.
original_triple_rows=[]; summarised_triple_rows=[]
for cluster_id,cached in original_cache.items():
    for t in cached["triples"]:
        original_triple_rows.append({"cluster_id":cluster_id,"representation":"original",**t})
for row in final_selected_summary_df.itertuples(index=False):
    summary_text = str(row.predicted_summary)
    if "SUMMARY_TRIPLE_CACHE" in globals() and summary_text in SUMMARY_TRIPLE_CACHE:
        _triples = SUMMARY_TRIPLE_CACHE[summary_text]
    else:
        _triples = extract_dependency_triples(summary_text)
        if "SUMMARY_TRIPLE_CACHE" in globals():
            SUMMARY_TRIPLE_CACHE[summary_text] = _triples
    for t in _triples:
        summarised_triple_rows.append({"cluster_id":row.cluster_id,"representation":"summarised",
            "summary_sentence_budget":REFERENCE_SUMMARY_LENGTH,"alpha_topic":REFERENCE_ALPHA,
            "beta_pattern":REFERENCE_BETA,"delta_redundancy":REFERENCE_DELTA,
            "n_topics":REFERENCE_N_TOPICS,**t})
original_triples_df=pd.DataFrame(original_triple_rows)
summarised_triples_df=pd.DataFrame(summarised_triple_rows)
triples_df=summarised_triples_df.copy()
del original_triple_rows, summarised_triple_rows
gc.collect()

#@title Build and export original-text and summarised-text RDF/OWL graphs
import importlib
import subprocess
import sys
import hashlib
import re

try:
    from rdflib import Graph, Literal, Namespace, URIRef
    from rdflib.namespace import OWL, RDF, RDFS
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "rdflib belum tersedia. Install sekali di cell terpisah: !pip install -q rdflib"
    ) from exc


for required in [
    "original_triples_df",
    "summarised_triples_df",
    "normalize_label",
]:
    if required not in globals():
        raise RuntimeError(
            f"{required} belum tersedia. Jalankan Original vs summarised "
            "triple extraction terlebih dahulu."
        )


EX = Namespace("https://example.org/paper2/")


def _rdf_safe_local_name(label, prefix):
    """Create a deterministic XML-NCName-safe URI local part.

    RDF/XML requires predicate URIs to be splittable into a namespace and a
    valid QName. Prefixing prevents numeric-only labels such as relation/1,
    while a short digest prevents collisions after character normalization.
    """
    raw = str(label or "").strip()
    normalized = str(normalize_label(raw) or "").strip()
    ascii_slug = re.sub(r"[^A-Za-z0-9._-]+", "_", normalized)
    ascii_slug = ascii_slug.strip("._-") or "value"
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()[:12]
    return f"{prefix}_{ascii_slug}_{digest}"


def build_rdf_graph(triples_frame, namespace=EX):
    graph = Graph()
    graph.bind("ex", namespace)
    graph.bind("rdf", RDF)
    graph.bind("rdfs", RDFS)
    graph.bind("owl", OWL)

    graph.add((namespace.Entity, RDF.type, OWL.Class))
    graph.add((namespace.Relation, RDF.type, OWL.ObjectProperty))
    graph.add((
        namespace.Entity,
        RDFS.label,
        Literal("Entity", lang="en"),
    ))
    graph.add((
        namespace.Relation,
        RDFS.label,
        Literal("Relation", lang="en"),
    ))

    skipped = 0
    uri_audit_rows = []

    for row in triples_frame.itertuples(index=False):
        subject_text = str(
            getattr(row, "subject", "") or ""
        ).strip()

        predicate_text = str(
            getattr(row, "predicate", "") or ""
        ).strip()

        object_text = str(
            getattr(row, "object", "") or ""
        ).strip()

        if not all([subject_text, predicate_text, object_text]):
            skipped += 1
            continue

        subject_id = _rdf_safe_local_name(subject_text, "ent")
        predicate_id = _rdf_safe_local_name(predicate_text, "rel")
        object_id = _rdf_safe_local_name(object_text, "ent")

        subject_uri = URIRef(namespace[f"entity/{subject_id}"])
        predicate_uri = URIRef(namespace[f"relation/{predicate_id}"])
        object_uri = URIRef(namespace[f"entity/{object_id}"])

        uri_audit_rows.extend([
            {"resource_type": "entity", "source_label": subject_text,
             "local_name": subject_id, "uri": str(subject_uri)},
            {"resource_type": "relation", "source_label": predicate_text,
             "local_name": predicate_id, "uri": str(predicate_uri)},
            {"resource_type": "entity", "source_label": object_text,
             "local_name": object_id, "uri": str(object_uri)},
        ])

        graph.add((
            subject_uri,
            RDF.type,
            namespace.Entity,
        ))

        graph.add((
            object_uri,
            RDF.type,
            namespace.Entity,
        ))

        graph.add((
            predicate_uri,
            RDF.type,
            OWL.ObjectProperty,
        ))

        graph.add((
            predicate_uri,
            RDFS.subPropertyOf,
            namespace.Relation,
        ))

        graph.add((
            subject_uri,
            predicate_uri,
            object_uri,
        ))

        graph.add((
            subject_uri,
            RDFS.label,
            Literal(subject_text, lang="en"),
        ))

        graph.add((
            object_uri,
            RDFS.label,
            Literal(object_text, lang="en"),
        ))

        graph.add((
            predicate_uri,
            RDFS.label,
            Literal(predicate_text, lang="en"),
        ))

    return graph, skipped, uri_audit_rows


g_original, skipped_original, original_uri_audit = build_rdf_graph(
    original_triples_df
)

g_summarised, skipped_summarised, summarised_uri_audit = build_rdf_graph(
    summarised_triples_df
)

uri_audit_df = (
    pd.DataFrame(original_uri_audit + summarised_uri_audit)
    .drop_duplicates(["resource_type", "source_label", "uri"])
    .sort_values(["resource_type", "source_label"], kind="stable")
    .reset_index(drop=True)
)
save_csv(uri_audit_df, OUT / "rdf_uri_mapping_audit.csv", index=False)

# Backward-compatible alias used by older notebook cells.
g = g_summarised


exports = [
    (
        g_original,
        OUT / "ontology_original.ttl",
        OUT / "ontology_original.rdf",
    ),
    (
        g_summarised,
        OUT / "ontology_summarised.ttl",
        OUT / "ontology_summarised.rdf",
    ),
]

for graph, ttl_path, rdf_path in exports:
    graph.serialize(
        destination=str(ttl_path),
        format="turtle",
    )

    try:
        graph.serialize(destination=str(rdf_path), format="xml")
    except ValueError as exc:
        raise RuntimeError(
            f"RDF/XML serialization failed for {rdf_path.name}. "
            "Inspect rdf_uri_mapping_audit.csv for an invalid predicate URI."
        ) from exc

    # Parse back to verify serialization integrity.
    check_ttl = Graph().parse(
        str(ttl_path),
        format="turtle",
    )

    check_rdf = Graph().parse(
        str(rdf_path),
        format="xml",
    )

    assert len(check_ttl) == len(graph)
    assert len(check_rdf) == len(graph)
    mirror_file(ttl_path)
    mirror_file(rdf_path)


ontology_comparison_df = pd.DataFrame([{
    "representation": "original",
    "source_relation_triples": len(original_triples_df),
    "skipped_malformed": skipped_original,
    "rdf_statements": len(g_original),
}, {
    "representation": "summarised",
    "source_relation_triples": len(summarised_triples_df),
    "skipped_malformed": skipped_summarised,
    "rdf_statements": len(g_summarised),
}])

save_csv(ontology_comparison_df, OUT / "ontology_size_comparison.csv", index=False,
)

print("=== Ontology/KG size comparison ===")
display(ontology_comparison_df)

print("Exported:")
print(" - ontology_original.ttl / ontology_original.rdf")
print(" - ontology_summarised.ttl / ontology_summarised.rdf")
print(" - ontology_size_comparison.csv")
print(" - rdf_uri_mapping_audit.csv")


## 11. Knowledge graph visualisation

In [ ]:
#@title Knowledge graph visualisation — sampled high-degree subgraph (max 50 edges)
import importlib
import subprocess
import sys
import pandas as pd

try:
    import networkx as nx
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "networkx belum tersedia. Install sekali: !pip install -q networkx"
    ) from exc

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "matplotlib belum tersedia. Install sekali: !pip install -q matplotlib"
    ) from exc


# ================================================================
# 1. INPUT VALIDATION
# ================================================================
if "triples_df" not in globals():
    _triples_checkpoint = checkpoint_path("triples_full.csv")
    if _triples_checkpoint.exists():
        triples_df = pd.read_csv(_triples_checkpoint)
        print(f"Loaded triples checkpoint for visualisation: {_triples_checkpoint} | rows={len(triples_df)}")
    else:
        raise RuntimeError(
            "triples_df belum tersedia dan checkpoint triples_full.csv belum ada. "
            "Jalankan Dependency-based triple extraction sekali terlebih dahulu."
        )

if triples_df.empty:
    raise RuntimeError(
        "triples_df kosong; tidak ada relation triple yang dapat divisualisasikan."
    )


# ================================================================
# 2. BUILD THE FULL KG FROM ALL EXTRACTED SUMMARY TRIPLES
#    IMPORTANT: the full graph is used for degree calculation.
#    Only the figure is sampled.
# ================================================================
FULL_KG = nx.DiGraph()

for row in triples_df.itertuples(index=False):
    subject = str(
        getattr(row, "subject", "") or ""
    ).strip()

    obj = str(
        getattr(row, "object", "") or ""
    ).strip()

    predicate = str(
        getattr(row, "predicate", "") or ""
    ).strip()

    if not subject or not obj or not predicate:
        continue

    if FULL_KG.has_edge(subject, obj):
        old_label = FULL_KG[subject][obj].get(
            "label",
            "",
        )

        existing_labels = [
            value.strip()
            for value in old_label.split(" / ")
            if value.strip()
        ]

        labels = list(
            dict.fromkeys(
                existing_labels + [predicate]
            )
        )

        FULL_KG[subject][obj]["label"] = (
            " / ".join(labels)
        )

    else:
        FULL_KG.add_edge(
            subject,
            obj,
            label=predicate,
        )


if FULL_KG.number_of_edges() == 0:
    raise RuntimeError(
        "Tidak ada triple valid setelah pembersihan; "
        "visualisasi tidak dibuat."
    )


# ================================================================
# 3. SAMPLE MAXIMUM 50 EDGES USING HIGHEST-DEGREE NODES
# ================================================================
MAX_VISUAL_EDGES = 50

# Degree = in-degree + out-degree for each node in the complete KG.
node_degree = dict(
    FULL_KG.degree()
)

# Rank every edge by the degree importance of its endpoints.
# This favours relations involving the most connected entities.
ranked_edges = sorted(
    FULL_KG.edges(data=True),
    key=lambda edge: (
        max(
            node_degree.get(edge[0], 0),
            node_degree.get(edge[1], 0),
        ),
        node_degree.get(edge[0], 0)
        + node_degree.get(edge[1], 0),
    ),
    reverse=True,
)

selected_edges = ranked_edges[
    :MAX_VISUAL_EDGES
]

KG = nx.DiGraph()

for source, target, data in selected_edges:
    KG.add_edge(
        source,
        target,
        **data,
    )

# Ranked node table for transparency/reproducibility.
node_degree_df = pd.DataFrame(
    [
        {
            "node": node,
            "degree_full_graph": degree,
            "included_in_visualisation":
                node in KG.nodes,
        }
        for node, degree
        in sorted(
            node_degree.items(),
            key=lambda item: item[1],
            reverse=True,
        )
    ]
)

save_csv(
    node_degree_df,
    OUT / "knowledge_graph_node_degrees.csv",
    index=False,
)


# ================================================================
# 4. VISUALISATION
# ================================================================
seed = (
    getattr(CFG, "seed", 42)
    if "CFG" in globals()
    else 42
)

node_count = KG.number_of_nodes()

fig_width = min(
    18,
    max(
        11,
        8 + node_count * 0.10,
    ),
)

fig, ax = plt.subplots(
    figsize=(fig_width, 9)
)

# Fewer iterations than the previous full-graph layout because this
# graph is explicitly limited to <= 50 edges.
pos = nx.spring_layout(
    KG,
    seed=seed,
    k=1.4,
    iterations=50,
)

nx.draw_networkx_nodes(
    KG,
    pos,
    node_size=650,
    node_color="#8ecae6",
    edgecolors="#1d3557",
    linewidths=0.7,
    alpha=0.9,
    ax=ax,
)

nx.draw_networkx_edges(
    KG,
    pos,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=13,
    edge_color="#6c757d",
    alpha=0.55,
    ax=ax,
)

nx.draw_networkx_labels(
    KG,
    pos,
    labels={
        node: str(node)[:42]
        for node in KG.nodes
    },
    font_size=7,
    ax=ax,
)

edge_labels = {
    (source, target):
        str(data.get("label", ""))[:32]
    for source, target, data
    in KG.edges(data=True)
}

nx.draw_networkx_edge_labels(
    KG,
    pos,
    edge_labels=edge_labels,
    font_size=6,
    rotate=False,
    label_pos=0.5,
    ax=ax,
)

ax.set_title(
    "Summary-derived Knowledge Graph\n"
    "Visualisation is a sampled subgraph "
    "(highest-degree nodes, max 50 edges)",
    fontsize=13,
    pad=16,
)

ax.axis("off")
fig.tight_layout()

figure_path = (
    OUT
    / "knowledge_graph_sampled_50_edges.png"
)

fig.savefig(
    figure_path,
    dpi=250,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)


# ================================================================
# 5. REPORTING / PROVENANCE
# ================================================================
visualisation_info_df = pd.DataFrame([{
    "full_graph_nodes":
        FULL_KG.number_of_nodes(),

    "full_graph_edges":
        FULL_KG.number_of_edges(),

    "visualised_nodes":
        KG.number_of_nodes(),

    "visualised_edges":
        KG.number_of_edges(),

    "max_visual_edges":
        MAX_VISUAL_EDGES,

    "sampling_method":
        (
            "edges ranked by endpoint degree "
            "in the complete summary-derived KG"
        ),

    "visualisation_is_sampled_subgraph":
        True,

    "rdf_export_scope":
        "FULL extracted triple set",
}])

save_csv(
    visualisation_info_df,
    OUT / "knowledge_graph_visualisation_info.csv",
    index=False,
)

print("\n" + "=" * 72)
print("KNOWLEDGE GRAPH VISUALISATION")
print("=" * 72)
print(
    "Full KG      : "
    f"{FULL_KG.number_of_nodes()} nodes | "
    f"{FULL_KG.number_of_edges()} edges"
)
print(
    "Visualised   : "
    f"{KG.number_of_nodes()} nodes | "
    f"{KG.number_of_edges()} edges"
)
print(
    "Sampling     : highest-degree-node relations, "
    "maximum 50 edges"
)
print(
    "INFO         : visualisation is a sampled subgraph"
)
print(
    "RDF/OWL      : remains exported from the FULL triple set "
    "in the preceding ontology-export cell"
)
print(
    "Figure saved :",
    figure_path,
)
print(
    "Metadata     :",
    OUT / "knowledge_graph_visualisation_info.csv",
)
print(
    "Node degrees :",
    OUT / "knowledge_graph_node_degrees.csv",
)
print("=" * 72)


## 12. WebNLG normalization and relation-extraction evaluation

In [ ]:
#@title WebNLG normalization and relation-extraction evaluation
import importlib
import subprocess
import sys
import pandas as pd
from tqdm.auto import tqdm

try:
    from datasets import load_dataset
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", "datasets>=3,<5",
    ])
    importlib.invalidate_caches()
    from datasets import load_dataset

for required_name in ["clean_text", "normalize_label", "extract_dependency_triples"]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} belum tersedia. Jalankan Dependency-based triple extraction terlebih dahulu."
        )

def load_webnlg_for_evaluation(limit):
    parquet_url = "https://huggingface.co/datasets/GEM/web_nlg/resolve/refs%2Fconvert%2Fparquet/en/test/0000.parquet"
    try:
        data = load_dataset("parquet", data_files={"test": parquet_url}, split=f"test[:{limit}]")
        print(f"WebNLG loaded from converted Parquet: {len(data)} records")
        return data
    except Exception as parquet_error:
        print("Converted WebNLG Parquet failed:", parquet_error)
    candidates = [
        ("GEM/web_nlg", None),
        ("web_nlg", "release_v3.0_en"),
        ("web_nlg", "release_v2"),
    ]
    errors = []
    for dataset_name, config_name in candidates:
        for split_name in ["test", "validation"]:
            try:
                split = f"{split_name}[:{limit}]"
                if config_name:
                    data = load_dataset(dataset_name, config_name, split=split)
                else:
                    data = load_dataset(dataset_name, split=split)
                print(
                    f"WebNLG loaded from {dataset_name} ({split_name}): {len(data)} records"
                )
                return data
            except Exception as exc:
                errors.append(f"{dataset_name}/{split_name}: {exc}")
    print("WebNLG could not be loaded. Last error:", errors[-1] if errors else "unknown")
    return None

if "webnlg_data" not in globals() or webnlg_data is None:
    limit = getattr(CFG, "max_webnlg", 250) if "CFG" in globals() else 250
    webnlg_data = load_webnlg_for_evaluation(limit)

if webnlg_data is None:
    raise RuntimeError(
        "WebNLG tidak berhasil dimuat. Periksa koneksi, lalu jalankan kembali sel ini."
    )

def webnlg_record_to_text_and_triples(record):
    text = ""
    gold = []

    # GEM schema: target is the verbalisation and meaning_representation is the triple list.
    for key in ["target", "text", "sentence"]:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            text = value
            break
        if isinstance(value, list) and value:
            text = str(value[0])
            break

    if not text and "lex" in record:
        lex = record["lex"]
        if isinstance(lex, dict):
            values = lex.get("text", lex.get("lex", []))
            text = values[0] if isinstance(values, list) and values else str(values)
        elif isinstance(lex, list) and lex:
            first = lex[0]
            text = str(first.get("lex", first.get("text", ""))) if isinstance(first, dict) else str(first)

    def walk(value):
        if isinstance(value, str) and "|" in value:
            parts = [part.strip() for part in value.split("|")]
            if len(parts) >= 3 and all(parts[:3]):
                gold.append(tuple(parts[:3]))
        elif isinstance(value, dict):
            subject = value.get("subject")
            predicate = value.get("property", value.get("predicate"))
            obj = value.get("object")
            if all(v is not None and str(v).strip() for v in [subject, predicate, obj]):
                gold.append((str(subject), str(predicate), str(obj)))
            else:
                for nested in value.values():
                    walk(nested)
        elif isinstance(value, (list, tuple)):
            for nested in value:
                walk(nested)

    for key in [
        "meaning_representation", "modified_triple_sets", "modifiedtripleset",
        "original_triple_sets", "originaltriplesets", "triples", "input",
    ]:
        if key in record:
            walk(record[key])
            if gold:
                break

    # Older WebNLG schema nests lexicalisations and triples inside entry.
    if "entry" in record and isinstance(record["entry"], dict):
        entry = record["entry"]
        if not text:
            lexs = entry.get("lexs", [])
            if lexs:
                text = str(lexs[0].get("lex", "")) if isinstance(lexs[0], dict) else str(lexs[0])
        if not gold:
            walk(entry.get("modifiedtripleset", []))

    # Preserve order while removing duplicate gold triples.
    gold = list(dict.fromkeys(gold))
    return clean_text(text), gold

def canon_triple(triple):
    return tuple(normalize_label(value) for value in triple[:3])

def comparable_relation(value):
    return "".join(ch for ch in normalize_label(value) if ch.isalnum())

def soft_match(predicted, gold):
    ps, pp, po = canon_triple(predicted)
    gs, gp, go = canon_triple(gold)
    pp_compact = comparable_relation(pp)
    gp_compact = comparable_relation(gp)
    if not all([ps, pp_compact, po, gs, gp_compact, go]):
        return False
    return (
        (ps in gs or gs in ps)
        and (po in go or go in po)
        and (pp_compact in gp_compact or gp_compact in pp_compact)
    )

def prf(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

web_rows = []
skipped_records = 0
_web_batch=int(getattr(CFG,"webnlg_batch_size",250))
for record in tqdm(webnlg_data, desc="Evaluating WebNLG relation extraction"):
    text, gold = webnlg_record_to_text_and_triples(record)
    if not text or not gold:
        skipped_records += 1
        continue

    predicted = [
        (item["subject"], item["predicate"], item["object"])
        for item in extract_dependency_triples(text)
    ]
    matched_gold = set()
    true_positive = 0
    for predicted_triple in predicted:
        hit = None
        for index, gold_triple in enumerate(gold):
            if index not in matched_gold and soft_match(predicted_triple, gold_triple):
                hit = index
                break
        if hit is not None:
            true_positive += 1
            matched_gold.add(hit)

    false_positive = len(predicted) - true_positive
    false_negative = len(gold) - true_positive
    precision, recall, f1 = prf(true_positive, false_positive, false_negative)
    web_rows.append({
        "n_gold": len(gold),
        "n_pred": len(predicted),
        "tp": true_positive,
        "fp": false_positive,
        "fn": false_negative,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })
    if len(web_rows) % _web_batch == 0:
        _tmp=pd.DataFrame(web_rows)
        save_csv(_tmp, OUT / "webnlg_relation_evaluation_checkpoint.csv", index=False)
        del _tmp
        gc.collect()

web_eval = pd.DataFrame(web_rows)
if web_eval.empty:
    raise RuntimeError(
        "Tidak ada record WebNLG yang berhasil dinormalisasi. Periksa schema dataset yang tercetak."
    )

# Report and export both macro and micro Precision / Recall / F1.
macro = web_eval[["precision", "recall", "f1"]].mean()

tp_total = int(web_eval["tp"].sum())
fp_total = int(web_eval["fp"].sum())
fn_total = int(web_eval["fn"].sum())

micro_p, micro_r, micro_f1 = prf(
    tp_total,
    fp_total,
    fn_total,
)

webnlg_aggregate = pd.DataFrame([{
    "evaluated_records": len(web_eval),
    "skipped_records": skipped_records,
    "tp_total": tp_total,
    "fp_total": fp_total,
    "fn_total": fn_total,

    "macro_precision": float(macro["precision"]),
    "macro_recall": float(macro["recall"]),
    "macro_f1": float(macro["f1"]),

    "micro_precision": float(micro_p),
    "micro_recall": float(micro_r),
    "micro_f1": float(micro_f1),
}])

print("=== WebNLG Macro Precision / Recall / F1 ===")
print(macro)

print("\n=== WebNLG Micro Precision / Recall / F1 ===")
print({
    "micro_precision": micro_p,
    "micro_recall": micro_r,
    "micro_f1": micro_f1,
})

print(
    f"\nEvaluated records: {len(web_eval)} | "
    f"Skipped records: {skipped_records}"
)

webnlg_path = OUT / "webnlg_relation_evaluation.csv"
webnlg_aggregate_path = OUT / "webnlg_relation_metrics_aggregate.csv"

save_csv(web_eval, webnlg_path, index=False)
save_csv(webnlg_aggregate, webnlg_aggregate_path, index=False)

print(f"Per-record WebNLG evaluation saved to: {webnlg_path}")
print(f"Aggregate WebNLG P/R/F1 saved to: {webnlg_aggregate_path}")


# Release dataset backing object after evaluation
try:
    del webnlg_data, web_rows
except Exception:
    pass
gc.collect(); print_ram("after WebNLG")


if "web_eval" in globals() and isinstance(web_eval, pd.DataFrame) and not web_eval.empty:
    save_checkpoint_csv(web_eval, "webnlg_eval.csv")


## 13. Original-vs-summarised ontology-driven QA on SQuAD

For each SQuAD validation context, two knowledge representations are built: (1) triples from the **original context**, and (2) triples from a **Paper-1-based compressed context**. Both are queried with the same QA retrieval function and evaluated against the same gold answers using EM and token-level F1. This is the downstream test of whether compression preserves useful knowledge.

In [ ]:
#@title LOW-MEMORY SQuAD QA — one configuration at a time + resume checkpoints
import importlib, re, subprocess, sys, gc, csv
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
try:
    from datasets import load_dataset
except ModuleNotFoundError:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","datasets>=3,<5"])
    importlib.invalidate_caches(); from datasets import load_dataset

def load_squad_for_qa(limit):
    urls=["https://huggingface.co/datasets/rajpurkar/squad/resolve/refs%2Fconvert%2Fparquet/plain_text/validation/0000.parquet",
          "https://huggingface.co/datasets/rajpurkar/squad/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet"]
    try: return load_dataset(getattr(CFG,"squad_dataset","rajpurkar/squad"),split=f"validation[:{limit}]")
    except Exception: pass
    for u in urls:
        try: return load_dataset("parquet",data_files={"validation":u},split=f"validation[:{limit}]")
        except Exception: pass
    raise RuntimeError("SQuAD could not be loaded")

def answer_from_triples(question, triples):
    if not triples: return ""
    texts=[f"{x['subject']} {x['predicate']} {x['object']}" for x in triples]
    try:
        X=TfidfVectorizer(stop_words="english").fit_transform([question,*texts])
        j=int(np.argmax(cosine_similarity(X[0:1],X[1:]).ravel())); del X
    except ValueError: j=0
    return triples[j]["subject"] if question.lower().strip().startswith("who was") else triples[j]["object"]
def normalize_answer(v): return " ".join(re.sub(r"[^a-z0-9 ]+"," ",str(v).lower()).split())
def token_f1(p,g):
    p=normalize_answer(p).split(); g=normalize_answer(g).split()
    if not p or not g: return float(p==g)
    common=sum((Counter(p)&Counter(g)).values())
    if not common:return 0.
    P=common/len(p); R=common/len(g); return 2*P*R/(P+R)
def best_em_f1(p,golds):
    return max(float(normalize_answer(p)==normalize_answer(g)) for g in golds), max(token_f1(p,g) for g in golds)

# Config list: Original + Paper-1 10/15 + every unique final Pareto candidate.
configs=[{"qa_configuration":"original","source":"original"}]
for L in [10,15]:
    configs.append({"qa_configuration":f"paper1_region_{L}","source":"Paper1_10_15_reference",
                    "alpha_topic":REFERENCE_ALPHA,"beta_pattern":REFERENCE_BETA,"delta_redundancy":REFERENCE_DELTA,
                    "summary_sentence_budget":L,"n_topics":REFERENCE_N_TOPICS})
for r in FINAL_PARETO_CONFIGS.drop_duplicates(subset=["alpha_topic","beta_pattern","delta_redundancy","summary_sentence_budget","n_topics"]).itertuples(index=False):
    configs.append({"qa_configuration":str(r.configuration_id),"source":"final_pareto",
                    "alpha_topic":float(r.alpha_topic),"beta_pattern":float(r.beta_pattern),"delta_redundancy":float(r.delta_redundancy),
                    "summary_sentence_budget":int(r.summary_sentence_budget),"n_topics":int(r.n_topics)})
# de-dup
uniq=[]; seen=set()
for c in configs:
    key=(c.get("alpha_topic"),c.get("beta_pattern"),c.get("delta_redundancy"),c.get("summary_sentence_budget"),c.get("n_topics"),c["source"])
    if key not in seen: seen.add(key); uniq.append(c)
configs=uniq

squad_data=load_squad_for_qa(int(CFG.max_squad))
qa_agg_rows=[]
qa_detail_dir = OUT / "qa_details"
qa_detail_dir.mkdir(parents=True, exist_ok=True)

for ci,cfgq in enumerate(configs):
    safe=re.sub(r"[^A-Za-z0-9_.-]+","_",cfgq["qa_configuration"])
    ck=OUT/f"qa_checkpoint_{safe}.csv"
    qa_detail_path = qa_detail_dir / f"squad_qa_details_{safe}.csv"
    if getattr(CFG,"checkpoint_resume",True) and ck.exists():
        qa_agg_rows.append(pd.read_csv(ck).iloc[0].to_dict()); print("QA resume:",cfgq["qa_configuration"]); continue
    sums={"em":0.,"f1":0.,"original_words":0,"repr_words":0,"n_triples":0.,"original_sent":0.,"repr_sent":0.}; n=0
    detail_buffer=[]
    context_cache={}  # one representation/triple extraction per unique context
    if qa_detail_path.exists(): qa_detail_path.unlink()  # remove an incomplete prior attempt
    for rec in tqdm(squad_data,desc=f"QA {cfgq['qa_configuration']}"):
        context=clean_text(rec.get("context","")); q=clean_text(rec.get("question","")); golds=rec.get("answers",{}).get("text",[])
        if not context or not q or not golds: continue
        if context in context_cache:
            rep, triples, ow, osent, rw, rsent = context_cache[context]
        else:
            original_sentences = safe_sentencize([context])
            ow = len(context.split()); osent = len(original_sentences)
            if cfgq["source"] == "original" or osent <= int(cfgq.get("summary_sentence_budget", osent)):
                rep = context
            else:
                rep,_=generate_summary([context],k=cfgq["summary_sentence_budget"],alpha=cfgq["alpha_topic"],beta=cfgq["beta_pattern"],
                                       delta=cfgq["delta_redundancy"],n_topics=cfgq["n_topics"])
            triples = extract_dependency_triples(rep)
            rw = len(rep.split()); rsent = len(safe_sentencize([rep]))
            context_cache[context] = (rep, triples, ow, osent, rw, rsent)
        pred=answer_from_triples(q,triples); em,f1=best_em_f1(pred,golds)
        n+=1
        sums["em"]+=em; sums["f1"]+=f1; sums["original_words"]+=ow; sums["repr_words"]+=rw
        sums["n_triples"]+=len(triples); sums["original_sent"]+=osent; sums["repr_sent"]+=rsent
        if getattr(CFG,"save_per_question_qa",True):
            detail_buffer.append({"id":rec.get("id"),"qa_configuration":cfgq["qa_configuration"],"source":cfgq["source"],
                                  "question":q,"gold":golds[0],"prediction":pred,"em":em,"f1":f1,"original_words":ow,
                                  "representation_words":rw,"n_triples":len(triples)})
            if len(detail_buffer)>=int(getattr(CFG,"qa_batch_size",250)):
                dfb=pd.DataFrame(detail_buffer); header=not qa_detail_path.exists(); dfb.to_csv(qa_detail_path,mode="a",header=header,index=False)
                mirror_file(qa_detail_path); detail_buffer=[]; del dfb; gc.collect()
        del triples, rep
    if detail_buffer:
        dfb=pd.DataFrame(detail_buffer); header=not qa_detail_path.exists(); dfb.to_csv(qa_detail_path,mode="a",header=header,index=False); mirror_file(qa_detail_path)
        del dfb, detail_buffer
    cr=sums["repr_words"]/sums["original_words"] if sums["original_words"] else 0.
    row={"qa_configuration":cfgq["qa_configuration"],"source":cfgq["source"],"n_questions":n,
         "em":sums["em"]/n if n else 0.,"f1":sums["f1"]/n if n else 0.,
         "mean_original_sentence_count":sums["original_sent"]/n if n else 0.,"mean_actual_representation_sentence_count":sums["repr_sent"]/n if n else 0.,
         "total_original_words":sums["original_words"],"total_representation_words":sums["repr_words"],
         "mean_n_triples":sums["n_triples"]/n if n else 0.,"corpus_compression_ratio":cr,"corpus_text_reduction_pct":100*(1-cr)}
    save_csv(pd.DataFrame([row]),ck,index=False); qa_agg_rows.append(row); context_cache.clear(); gc.collect(); print_ram(f"after QA {cfgq['qa_configuration']}")

qa_aggregate=pd.DataFrame(qa_agg_rows)
orig=qa_aggregate[qa_aggregate.qa_configuration=="original"].iloc[0]
qa_aggregate["delta_em_vs_original"]=qa_aggregate.em-float(orig.em)
qa_aggregate["delta_f1_vs_original"]=qa_aggregate.f1-float(orig.f1)
save_csv(qa_aggregate,OUT/"squad_qa_pareto_candidates_aggregate.csv",index=False)
qa_comparison_df=pd.DataFrame()  # details live on disk; keep RAM small
qa_df=qa_comparison_df
print("=== QA aggregate ==="); display(qa_aggregate)
try: del squad_data
except Exception: pass
gc.collect(); print_ram("after SQuAD")


if "FINAL_PARETO_CONFIGS" in globals() and isinstance(FINAL_PARETO_CONFIGS, pd.DataFrame) and not FINAL_PARETO_CONFIGS.empty:
    save_checkpoint_csv(FINAL_PARETO_CONFIGS, "final_pareto_configurations.csv")


if "qa_comparison_df" in globals() and isinstance(qa_comparison_df, pd.DataFrame) and not qa_comparison_df.empty:
    save_checkpoint_csv(qa_comparison_df, "squad_qa_detail.csv")
if "qa_aggregate" in globals() and isinstance(qa_aggregate, pd.DataFrame) and not qa_aggregate.empty:
    save_checkpoint_csv(qa_aggregate, "squad_qa_aggregate.csv")


## 14. Ablation study

In [ ]:
#@title OPTIONAL — Ablation study using cached features
import gc
import pandas as pd

if "REFERENCE_N_TOPICS" not in globals():
    raise RuntimeError("Run STEP 4 first.")

ablations = [
    "full",
    "no_topic",
    "no_pattern",
    "no_redundancy",
]

rows = []

for mode in ablations:
    rouge_metric_names = [m for m in CFG.saved_metrics if m.startswith("rouge")]
    sums = {metric: 0.0 for metric in rouge_metric_names}
    n = 0

    for idx, row in enumerate(summary_data, start=1):
        features = get_cached_summary_features(
            row["cluster_id"],
            row["documents"],
            REFERENCE_N_TOPICS,
        )

        pred, _ = generate_summary_from_features(
            features,
            k=REFERENCE_SUMMARY_LENGTH,
            ablation=mode,
            alpha=REFERENCE_ALPHA,
            beta=REFERENCE_BETA,
            delta=REFERENCE_DELTA,
            return_detail=False,
        )

        rouge_values = compute_all_rouge(row["reference_summary"], pred)
        for metric in rouge_metric_names:
            sums[metric] += rouge_values[metric]
        n += 1

        if idx % 5 == 0:
            cleanup_memory()

    rows.append({
        "configuration": mode,
        "n_clusters": n,
        **{metric: sums[metric] / n for metric in rouge_metric_names},
    })

ablation_df = pd.DataFrame(rows)

save_csv(
    ablation_df,
    OUT / "ablation_study_all_rouge.csv",
    index=False,
)

display(ablation_df)
print_ram("after ablation")


## 15. Automatically generate paper-ready tables and figures

In [ ]:
#@title Consolidate single-scenario Pareto results for Paper 2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Reporting only: load saved v2.2 outputs when variables are not in memory.
scenario_path = scenario_result_file()
pareto_path = OUT / f"final_pareto_configurations_{active_experiment_split()}.csv"

if "all_scenarios" not in globals():
    if not scenario_path.exists():
        raise RuntimeError(
            "No single-scenario results found. Run/merge all scenarios, then run "
            "COMBINE ALL SAVED SINGLE-SCENARIO RESULTS + PARETO."
        )
    all_scenarios = pd.read_csv(scenario_path)

if "FINAL_PARETO_CONFIGS" not in globals():
    if not pareto_path.exists():
        raise RuntimeError(
            "Pareto results not found. Run COMBINE ALL SAVED "
            "SINGLE-SCENARIO RESULTS + PARETO first."
        )
    FINAL_PARETO_CONFIGS = pd.read_csv(pareto_path)

pareto_ids = set(FINAL_PARETO_CONFIGS["configuration_id"].astype(str))
report_scenarios = all_scenarios.copy()
report_scenarios["pareto_optimal"] = (
    report_scenarios["configuration_id"].astype(str).isin(pareto_ids)
)

display_columns = [c for c in [
    "configuration_id", "alpha_topic", "beta_pattern",
    "delta_redundancy", "summary_sentence_budget", "n_topics",
    *CFG.saved_metrics,
    "runtime_minutes", "pareto_optimal",
] if c in report_scenarios.columns]

print("=== ALL SAVED SINGLE-SCENARIO RESULTS ===")
print("Rows:", len(report_scenarios))
print("Unique configuration IDs:", report_scenarios["configuration_id"].nunique())
display(report_scenarios[display_columns])

print("\n=== FINAL PARETO CONFIGURATIONS ===")
print("Pareto configurations:", len(FINAL_PARETO_CONFIGS))
display(FINAL_PARETO_CONFIGS)

# One v2.2 trade-off figure: all configurations, with Pareto points emphasized.
if {"compression_gain", "preservation_f1"}.issubset(report_scenarios.columns):
    dominated = report_scenarios[~report_scenarios["pareto_optimal"]]
    pareto = report_scenarios[report_scenarios["pareto_optimal"]]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(dominated["compression_gain"], dominated["preservation_f1"],
               alpha=0.30, label="Dominated")
    ax.scatter(pareto["compression_gain"], pareto["preservation_f1"],
               marker="x", s=80, label="Pareto")
    ax.set_xlabel("Compression Gain")
    ax.set_ylabel("Triple Preservation F1")
    ax.set_title("Single-Scenario Pareto Trade-off")
    ax.legend()
    plt.tight_layout()
    figure_path = OUT / "pareto_compression_vs_tripleF1.png"
    plt.savefig(figure_path, dpi=250, bbox_inches="tight")
    plt.show()
    mirror_file(figure_path)

# Optional downstream results: report only when their cells have run.
if "qa_aggregate" in globals() and isinstance(qa_aggregate, pd.DataFrame) and not qa_aggregate.empty:
    print("\n=== DOWNSTREAM QA ===")
    display(qa_aggregate)
else:
    print("\nQA aggregate not available yet; skipped in this report.")

relation_metrics = {}
if "web_eval" in globals() and isinstance(web_eval, pd.DataFrame) and not web_eval.empty:
    print("\n=== WEBNLG RELATION EXTRACTION ===")
    display(web_eval.head())
    if {"precision", "recall", "f1"}.issubset(web_eval.columns):
        means = web_eval[["precision", "recall", "f1"]].mean()
        relation_metrics = {
            "macro_precision": float(means["precision"]),
            "macro_recall": float(means["recall"]),
            "macro_f1": float(means["f1"]),
        }
else:
    print("WebNLG evaluation not available yet; skipped in this report.")

reference_metadata = {}
if all(name in globals() for name in [
    "REFERENCE_ALPHA", "REFERENCE_BETA", "REFERENCE_DELTA",
    "REFERENCE_SUMMARY_LENGTH", "REFERENCE_N_TOPICS",
]):
    reference_metadata = {
        "alpha_topic": REFERENCE_ALPHA,
        "beta_pattern": REFERENCE_BETA,
        "delta_redundancy": REFERENCE_DELTA,
        "summary_sentence_budget": REFERENCE_SUMMARY_LENGTH,
        "n_topics": REFERENCE_N_TOPICS,
        "note": "Deterministic downstream reference from the Pareto set; not globally best.",
    }

metrics_report = {
    "selection_method": "Pareto non-dominated multi-metric selection",
    "saved_metrics": list(CFG.saved_metrics),
    "pareto_metrics": list(CFG.pareto_metrics),
    "n_saved_scenarios": int(len(report_scenarios)),
    "n_unique_configurations": int(report_scenarios["configuration_id"].nunique()),
    "n_pareto_configurations": int(len(FINAL_PARETO_CONFIGS)),
    "reference_candidate": reference_metadata,
    "webnlg_independent_validation": relation_metrics,
}

save_json(metrics_report, OUT / "metrics.json", indent=2)
save_csv(report_scenarios, OUT / "table_all_single_scenarios.csv", index=False)
save_csv(FINAL_PARETO_CONFIGS, OUT / "table_final_pareto_configurations.csv", index=False)

latex_path = OUT / "table_final_pareto_configurations.tex"
save_text(FINAL_PARETO_CONFIGS.to_latex(index=False, float_format="%.4f"), latex_path)

print("\nReporting complete. No weighted composite objective was used.")
print("Main output: table_final_pareto_configurations.csv")


## 16. Package all outputs into a ZIP

In [ ]:

import shutil
zip_path=shutil.make_archive("/content/paper2_outputs","zip",OUT)
print("Created:",zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download manually from the Files panel:",zip_path)
